In [1]:
# ============================================================
# Baseline Portion B0: Setup and config
# Model: ResNet18 + Bengali Transformer + Concatenation Fusion
# ============================================================

!pip install -q scikit-learn tqdm pillow matplotlib pandas

import os
import re
import gc
import json
import math
import time
import random
import warnings
from pathlib import Path
from dataclasses import dataclass, asdict
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image, ImageFile

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

ImageFile.LOAD_TRUNCATED_IMAGES = True
Image.MAX_IMAGE_PIXELS = None


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------
BASE_SEED = 42

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Keep benchmark=True for speed on Kaggle GPUs.
    # For strict determinism, set deterministic=True and benchmark=False.
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(BASE_SEED)


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
PROJECT_DIR = Path("/kaggle/working/tinyagrivqa_bn") if Path("/kaggle/working").exists() else Path("./tinyagrivqa_bn")

P2_DIR = PROJECT_DIR / "portion2_architecture"
P2_DATA_DIR = P2_DIR / "data"

BASE_DIR = PROJECT_DIR / "portion_baseline_resnet18_concat"
BASE_CKPT_DIR = BASE_DIR / "checkpoints"
BASE_LOG_DIR = BASE_DIR / "logs"
BASE_REPORT_DIR = BASE_DIR / "reports"
BASE_PRED_DIR = BASE_DIR / "predictions"

for d in [BASE_DIR, BASE_CKPT_DIR, BASE_LOG_DIR, BASE_REPORT_DIR, BASE_PRED_DIR]:
    d.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Device and hardware profile
# ------------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_GPUS = torch.cuda.device_count()

GPU_NAMES = []
if torch.cuda.is_available():
    for i in range(NUM_GPUS):
        GPU_NAMES.append(torch.cuda.get_device_name(i))

def detect_hardware_profile():
    if not torch.cuda.is_available():
        return "cpu"

    name = " ".join(GPU_NAMES).lower()

    if NUM_GPUS >= 2 and "t4" in name:
        return "t4x2"
    if "p100" in name:
        return "p100"
    if "t4" in name:
        return "t4_single"
    if "v100" in name:
        return "v100"
    if "a100" in name:
        return "a100"

    return "cuda_generic"

HW_PROFILE = detect_hardware_profile()

print("=" * 90)
print("BASELINE SETUP: ResNet18 + Bengali Transformer + Concat Fusion")
print("=" * 90)
print("Project dir:", PROJECT_DIR)
print("Baseline dir:", BASE_DIR)
print("Device:", DEVICE)
print("GPUs:", NUM_GPUS)
print("GPU names:", GPU_NAMES)
print("Hardware profile:", HW_PROFILE)


# ------------------------------------------------------------
# Baseline config
# ------------------------------------------------------------
@dataclass
class BaselineConfig:
    # Data
    image_size: int = 224
    max_text_len: int = 48

    # Model
    image_backbone: str = "resnet18"
    image_pretrained: bool = False
    text_dim: int = 256
    text_layers: int = 3
    text_heads: int = 4
    text_ff_dim: int = 768
    fusion_dim: int = 512
    dropout: float = 0.10

    # Training
    epochs: int = 12
    train_batch_size: int = 64
    eval_batch_size: int = 128
    gradient_accumulation_steps: int = 1
    num_workers: int = 2
    pin_memory: bool = True

    # Optimization
    lr: float = 2e-4
    image_encoder_lr: float = 5e-5
    weight_decay: float = 1e-4
    warmup_ratio: float = 0.06
    max_grad_norm: float = 1.0

    # Loss
    label_smoothing: float = 0.02

    # AMP
    mixed_precision: str = "fp16"

    # Early stopping
    early_stop: bool = True
    early_stopping_patience: int = 3
    min_delta: float = 0.001
    monitor_metric: str = "answer_macro_f1"

    # Debug controls
    max_train_batches: int | None = None
    max_val_batches: int | None = None

    # Checkpointing
    save_every_epoch: bool = False


BASE_CFG = BaselineConfig()

# Hardware-specific safe defaults
if HW_PROFILE == "t4x2":
    BASE_CFG.train_batch_size = 64
    BASE_CFG.eval_batch_size = 128
    BASE_CFG.num_workers = 2

elif HW_PROFILE == "p100":
    BASE_CFG.train_batch_size = 48
    BASE_CFG.eval_batch_size = 96
    BASE_CFG.num_workers = 2

elif HW_PROFILE == "t4_single":
    BASE_CFG.train_batch_size = 32
    BASE_CFG.eval_batch_size = 64
    BASE_CFG.num_workers = 2

elif HW_PROFILE == "cpu":
    BASE_CFG.train_batch_size = 8
    BASE_CFG.eval_batch_size = 16
    BASE_CFG.num_workers = 0
    BASE_CFG.mixed_precision = "no"

else:
    BASE_CFG.train_batch_size = 48
    BASE_CFG.eval_batch_size = 96
    BASE_CFG.num_workers = 2


with open(BASE_REPORT_DIR / "baseline_resnet18_concat_config.json", "w", encoding="utf-8") as f:
    json.dump(asdict(BASE_CFG), f, ensure_ascii=False, indent=2)

print("\nBaseline config:")
print(json.dumps(asdict(BASE_CFG), indent=2, ensure_ascii=False))

BASELINE SETUP: ResNet18 + Bengali Transformer + Concat Fusion
Project dir: /kaggle/working/tinyagrivqa_bn
Baseline dir: /kaggle/working/tinyagrivqa_bn/portion_baseline_resnet18_concat
Device: cuda
GPUs: 2
GPU names: ['Tesla T4', 'Tesla T4']
Hardware profile: t4x2

Baseline config:
{
  "image_size": 224,
  "max_text_len": 48,
  "image_backbone": "resnet18",
  "image_pretrained": false,
  "text_dim": 256,
  "text_layers": 3,
  "text_heads": 4,
  "text_ff_dim": 768,
  "fusion_dim": 512,
  "dropout": 0.1,
  "epochs": 12,
  "train_batch_size": 64,
  "eval_batch_size": 128,
  "gradient_accumulation_steps": 1,
  "num_workers": 2,
  "pin_memory": true,
  "lr": 0.0002,
  "image_encoder_lr": 5e-05,
  "weight_decay": 0.0001,
  "warmup_ratio": 0.06,
  "max_grad_norm": 1.0,
  "label_smoothing": 0.02,
  "mixed_precision": "fp16",
  "early_stop": true,
  "early_stopping_patience": 3,
  "min_delta": 0.001,
  "monitor_metric": "answer_macro_f1",
  "max_train_batches": null,
  "max_val_batches": null,


In [2]:
# ============================================================
# FIXED B1-R2: Build notebook-compatible P2 artifacts from HF
# Dataset: SyedNazmusSakib/PlantVillageVQA
# Use before ResNet18 + Transformer concat baseline B2
# ============================================================

print("=" * 90)
print("B1-R2: BUILDING P2 ARTIFACTS LIKE THE GIVEN NOTEBOOK")
print("=" * 90)

import os
import re
import gc
import json
import zipfile
import random
import hashlib
import subprocess
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

try:
    from sklearn.model_selection import train_test_split
except Exception:
    subprocess.run(["pip", "install", "-q", "scikit-learn"], check=True)
    from sklearn.model_selection import train_test_split

try:
    from huggingface_hub import snapshot_download
except Exception:
    subprocess.run(["pip", "install", "-q", "huggingface_hub"], check=True)
    from huggingface_hub import snapshot_download


# ------------------------------------------------------------
# Seed
# ------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
dataset_id = "SyedNazmusSakib/PlantVillageVQA"

PROJECT_DIR = Path("/kaggle/working/tinyagrivqa_bn") if Path("/kaggle/working").exists() else Path("./tinyagrivqa_bn")

DATA_DIR = PROJECT_DIR / "data_bn"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
REPORT_DIR = PROJECT_DIR / "reports"
AUDIT_DIR = REPORT_DIR / "audit"

P2_DIR = PROJECT_DIR / "portion2_architecture"
P2_DATA_DIR = P2_DIR / "data"
P2_MODEL_DIR = P2_DIR / "models"
P2_REPORT_DIR = P2_DIR / "reports"

EXTRACT_DIR = PROJECT_DIR / "PlantVillageVQA_extracted"
HF_CACHE_DIR = PROJECT_DIR / "cache" / "hf_snapshot"

for d in [
    PROJECT_DIR, DATA_DIR, INTERIM_DIR, PROCESSED_DIR, REPORT_DIR, AUDIT_DIR,
    P2_DIR, P2_DATA_DIR, P2_MODEL_DIR, P2_REPORT_DIR, EXTRACT_DIR, HF_CACHE_DIR
]:
    d.mkdir(parents=True, exist_ok=True)

P2_TRAIN_CSV = P2_DATA_DIR / "p2_train.csv"
P2_VAL_CSV = P2_DATA_DIR / "p2_val.csv"
P2_TEST_CSV = P2_DATA_DIR / "p2_test.csv"
P2_FULL_CSV = P2_DATA_DIR / "p2_model_table.csv"
P2_MAPPING_JSON = P2_DATA_DIR / "p2_label_mapping.json"
P2_TOKENIZER_JSON = P2_DATA_DIR / "p2_bengali_word_tokenizer.json"


# ------------------------------------------------------------
# Download/extract PlantVillageVQA.zip
# ------------------------------------------------------------
print("Searching/downloading Hugging Face dataset:", dataset_id)

existing_zips = []
for root in [PROJECT_DIR, HF_CACHE_DIR, Path("/kaggle/working"), Path("/kaggle/input"), Path.home() / ".cache" / "huggingface"]:
    if root.exists():
        existing_zips.extend(list(root.rglob("PlantVillageVQA.zip")))

existing_zips = sorted(set(existing_zips), key=lambda p: p.stat().st_size if p.exists() else 0, reverse=True)

if existing_zips:
    zip_path = existing_zips[0]
else:
    snapshot_dir = snapshot_download(
        repo_id=dataset_id,
        repo_type="dataset",
        local_dir=str(HF_CACHE_DIR),
        allow_patterns=["PlantVillageVQA.zip", "README.md"],
    )
    snapshot_dir = Path(snapshot_dir)
    zip_candidates = list(snapshot_dir.rglob("PlantVillageVQA.zip"))
    if not zip_candidates:
        raise FileNotFoundError("PlantVillageVQA.zip not found after snapshot_download.")
    zip_path = zip_candidates[0]

print("ZIP path:", zip_path)
print("ZIP size GB:", round(zip_path.stat().st_size / (1024 ** 3), 3))

csv_candidates = list(EXTRACT_DIR.rglob("PlantVillageVQA.csv"))
if not csv_candidates:
    print("Extracting ZIP to:", EXTRACT_DIR)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(EXTRACT_DIR)
else:
    print("Dataset already extracted.")

csv_candidates = list(EXTRACT_DIR.rglob("PlantVillageVQA.csv"))
if not csv_candidates:
    raise FileNotFoundError(f"PlantVillageVQA.csv not found under {EXTRACT_DIR}")

ANN_PATH = csv_candidates[0]
print("Annotation CSV:", ANN_PATH)


# ------------------------------------------------------------
# Load annotation
# ------------------------------------------------------------
ann_df = pd.read_csv(ANN_PATH)

required_cols = ["image_id", "question_type", "question", "answer", "image_path", "split"]
missing_cols = [c for c in required_cols if c not in ann_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}. Available: {ann_df.columns.tolist()}")

def norm_space(x):
    if pd.isna(x):
        return ""
    return re.sub(r"\s+", " ", str(x)).strip()

for c in required_cols:
    ann_df[c] = ann_df[c].map(norm_space)

print("Raw annotation shape:", ann_df.shape)
print("Raw split counts:")
print(ann_df["split"].value_counts(dropna=False))


# ------------------------------------------------------------
# Resolve image paths like notebook: image_id -> actual basename
# ------------------------------------------------------------
print("\nResolving image paths by image_id basename...")

valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
image_files = [p for p in EXTRACT_DIR.rglob("*") if p.is_file() and p.suffix.lower() in valid_exts]

if len(image_files) == 0:
    raise FileNotFoundError(f"No image files found under {EXTRACT_DIR}")

basename_to_path = {}
for p in image_files:
    basename_to_path.setdefault(p.name, str(p))

ann_df["resolved_image_path"] = ann_df["image_id"].map(basename_to_path)

# Fallback by image_path basename
missing_mask = ann_df["resolved_image_path"].isna()
if missing_mask.any():
    ann_df.loc[missing_mask, "resolved_image_path"] = (
        ann_df.loc[missing_mask, "image_path"]
        .map(lambda x: basename_to_path.get(Path(str(x)).name, np.nan))
    )

missing_count = int(ann_df["resolved_image_path"].isna().sum())
print("Image files found:", len(image_files))
print("Unique image basenames:", len(basename_to_path))
print("Missing image rows after basename resolution:", missing_count)

if missing_count > 0:
    missing_report = ann_df.loc[
        ann_df["resolved_image_path"].isna(),
        ["image_id", "image_path", "split"]
    ].drop_duplicates()
    missing_report.to_csv(REPORT_DIR / "missing_image_path_report_baseline.csv", index=False)
    print("Saved missing image report:", REPORT_DIR / "missing_image_path_report_baseline.csv")

ann_df = ann_df.dropna(subset=["resolved_image_path"]).reset_index(drop=True)

if len(ann_df) == 0:
    raise RuntimeError(
        "All rows were dropped because images were unresolved. "
        "Check whether image_id matches extracted image basenames."
    )

print("Rows after image resolution:", len(ann_df))
print("Unique resolved images:", ann_df["image_id"].nunique())


# ------------------------------------------------------------
# Notebook-style crop/disease inference
# ------------------------------------------------------------
CROP_BN = {
    "apple": "আপেল",
    "blueberry": "ব্লুবেরি",
    "cherry": "চেরি",
    "grape": "আঙুর",
    "orange": "কমলা",
    "peach": "পিচ",
    "pepper_bell": "বেল পেপার",
    "potato": "আলু",
    "raspberry": "রাস্পবেরি",
    "soybean": "সয়াবিন",
    "squash": "স্কোয়াশ",
    "strawberry": "স্ট্রবেরি",
    "tomato": "টমেটো",
    "corn_maize": "ভুট্টা",
    "background": "ব্যাকগ্রাউন্ড",
    "unknown": "অজানা",
}

DISEASE_BN = {
    "apple_scab": "আপেল স্ক্যাব",
    "apple_black_rot": "আপেলের ব্ল্যাক রট",
    "cedar_apple_rust": "সিডার আপেল রাস্ট",
    "cherry_powdery_mildew": "চেরির পাউডারি মিলডিউ",
    "grape_black_rot": "আঙুরের ব্ল্যাক রট",
    "grape_esca_black_measles": "আঙুরের এস্কা বা ব্ল্যাক মিজলস",
    "grape_leaf_blight_isariopsis_leaf_spot": "আঙুরের লিফ ব্লাইট বা আইসারিওপসিস লিফ স্পট",
    "orange_huanglongbing_citrus_greening": "কমলার হুয়াংলংবিং বা সাইট্রাস গ্রিনিং",
    "peach_bacterial_spot": "পিচের ব্যাকটেরিয়াল স্পট",
    "pepper_bell_bacterial_spot": "বেল পেপারের ব্যাকটেরিয়াল স্পট",
    "potato_early_blight": "আলুর আর্লি ব্লাইট",
    "potato_late_blight": "আলুর লেট ব্লাইট",
    "squash_powdery_mildew": "স্কোয়াশের পাউডারি মিলডিউ",
    "strawberry_leaf_scorch": "স্ট্রবেরির লিফ স্কর্চ",
    "tomato_bacterial_spot": "টমেটোর ব্যাকটেরিয়াল স্পট",
    "tomato_early_blight": "টমেটোর আর্লি ব্লাইট",
    "tomato_late_blight": "টমেটোর লেট ব্লাইট",
    "tomato_leaf_mold": "টমেটোর লিফ মোল্ড",
    "tomato_septoria_leaf_spot": "টমেটোর সেপ্টোরিয়া লিফ স্পট",
    "tomato_spider_mites_two_spotted_spider_mite": "টমেটোর টু-স্পটেড স্পাইডার মাইট",
    "tomato_target_spot": "টমেটোর টার্গেট স্পট",
    "tomato_yellow_leaf_curl_virus": "টমেটোর ইয়েলো লিফ কার্ল ভাইরাস",
    "tomato_mosaic_virus": "টমেটোর মোজাইক ভাইরাস",
    "corn_cercospora_leaf_spot_gray_leaf_spot": "ভুট্টার সারকোস্পোরা লিফ স্পট বা গ্রে লিফ স্পট",
    "corn_common_rust": "ভুট্টার কমন রাস্ট",
    "corn_northern_leaf_blight": "ভুট্টার নর্দার্ন লিফ ব্লাইট",
    "healthy": "সুস্থ",
    "no_leaf": "পাতা নেই",
    "unknown": "অজানা",
}

DISEASE_TO_CROP = {
    "apple_scab": "apple",
    "apple_black_rot": "apple",
    "cedar_apple_rust": "apple",
    "cherry_powdery_mildew": "cherry",
    "grape_black_rot": "grape",
    "grape_esca_black_measles": "grape",
    "grape_leaf_blight_isariopsis_leaf_spot": "grape",
    "orange_huanglongbing_citrus_greening": "orange",
    "peach_bacterial_spot": "peach",
    "pepper_bell_bacterial_spot": "pepper_bell",
    "potato_early_blight": "potato",
    "potato_late_blight": "potato",
    "squash_powdery_mildew": "squash",
    "strawberry_leaf_scorch": "strawberry",
    "tomato_bacterial_spot": "tomato",
    "tomato_early_blight": "tomato",
    "tomato_late_blight": "tomato",
    "tomato_leaf_mold": "tomato",
    "tomato_septoria_leaf_spot": "tomato",
    "tomato_spider_mites_two_spotted_spider_mite": "tomato",
    "tomato_target_spot": "tomato",
    "tomato_yellow_leaf_curl_virus": "tomato",
    "tomato_mosaic_virus": "tomato",
    "corn_cercospora_leaf_spot_gray_leaf_spot": "corn_maize",
    "corn_common_rust": "corn_maize",
    "corn_northern_leaf_blight": "corn_maize",
}

def norm_en(x):
    x = "" if pd.isna(x) else str(x).lower()
    x = x.replace("_", " ")
    x = re.sub(r"[^a-z0-9\s\-\(\),./]", " ", x)
    return re.sub(r"\s+", " ", x).strip()

def regex_any(text, patterns):
    return any(re.search(p, text, flags=re.IGNORECASE) for p in patterns)

STRICT_CROP_PATTERNS = {
    "tomato": [r"\btomato\s+(leaf|leaves|plant|crop)\b", r"\btomato\b"],
    "potato": [r"\bpotato\s+(leaf|leaves|plant|crop)\b", r"\bpotato\b"],
    "corn_maize": [r"\b(corn|maize)\s+(leaf|leaves|plant|crop)\b", r"\bcorn\b", r"\bmaize\b"],
    "apple": [r"\bapple\s+(leaf|leaves|plant|tree|crop)\b", r"\bapple\b"],
    "grape": [r"\bgrape\s+(leaf|leaves|plant|vine|crop)\b", r"\bgrape\b"],
    "orange": [r"\b(citrus|orange)\s+(leaf|leaves|plant|tree|crop)\b", r"\bcitrus\b"],
    "peach": [r"\bpeach\s+(leaf|leaves|plant|tree|crop)\b", r"\bpeach\b"],
    "pepper_bell": [r"\b(bell\s+pepper|pepper)\s+(leaf|leaves|plant|crop)\b", r"\bpepper\b"],
    "cherry": [r"\bcherry\s+(leaf|leaves|plant|tree|crop)\b", r"\bcherry\b"],
    "blueberry": [r"\bblueberry\b"],
    "raspberry": [r"\braspberry\b"],
    "soybean": [r"\b(soybean|soy)\b"],
    "squash": [r"\bsquash\b"],
    "strawberry": [r"\bstrawberry\b"],
}

SPECIFIC_DISEASE_RULES = [
    ("tomato_yellow_leaf_curl_virus", [r"\btylcv\b", r"yellow\s+leaf\s+curl", r"leaf\s+curl\s+virus"]),
    ("orange_huanglongbing_citrus_greening", [r"\bhlb\b", r"huanglongbing", r"citrus\s+greening"]),
    ("tomato_mosaic_virus", [r"tomato\s+mosaic", r"\bmosaic\s+virus\b"]),
    ("tomato_septoria_leaf_spot", [r"septoria"]),
    ("tomato_spider_mites_two_spotted_spider_mite", [r"two[- ]spotted\s+spider\s+mite", r"spider\s+mites?"]),
    ("tomato_target_spot", [r"target\s+spot"]),
    ("tomato_leaf_mold", [r"leaf\s+mold", r"leaf\s+mould"]),
    ("corn_cercospora_leaf_spot_gray_leaf_spot", [r"cercospora", r"gray\s+leaf\s+spot", r"grey\s+leaf\s+spot"]),
    ("corn_common_rust", [r"common\s+rust"]),
    ("corn_northern_leaf_blight", [r"northern\s+leaf\s+blight"]),
    ("grape_esca_black_measles", [r"\besca\b", r"black\s+measles"]),
    ("grape_leaf_blight_isariopsis_leaf_spot", [r"isariopsis", r"grape.*leaf\s+blight"]),
    ("strawberry_leaf_scorch", [r"leaf\s+scorch"]),
    ("cedar_apple_rust", [r"cedar\s+apple\s+rust"]),
]

AMBIGUOUS_DISEASE_RULES = [
    (r"late\s+blight", {"tomato": "tomato_late_blight", "potato": "potato_late_blight", "default": "potato_late_blight"}),
    (r"early\s+blight", {"tomato": "tomato_early_blight", "potato": "potato_early_blight", "default": "potato_early_blight"}),
    (r"bacterial\s+spot", {"tomato": "tomato_bacterial_spot", "pepper_bell": "pepper_bell_bacterial_spot", "peach": "peach_bacterial_spot", "default": "tomato_bacterial_spot"}),
    (r"powdery\s+mildew", {"cherry": "cherry_powdery_mildew", "squash": "squash_powdery_mildew", "default": "squash_powdery_mildew"}),
    (r"black\s+rot", {"apple": "apple_black_rot", "grape": "grape_black_rot", "default": "apple_black_rot"}),
    (r"\bapple\s+scab\b|\bscab\b", {"apple": "apple_scab", "default": "apple_scab"}),
]

def infer_image_labels(g):
    crop_votes = Counter()
    disease_votes = Counter()

    all_text = " ".join(g["answer"].fillna("").astype(str).tolist())
    text = norm_en(all_text)

    if regex_any(text, [r"background\s+image", r"no\s+plant\s+leaf", r"without\s+a\s+plant\s+leaf"]):
        return "background", "no_leaf", "background_no_leaf"

    for crop, patterns in STRICT_CROP_PATTERNS.items():
        if regex_any(text, patterns):
            crop_votes[crop] += 5

    for disease, patterns in SPECIFIC_DISEASE_RULES:
        if regex_any(text, patterns):
            disease_votes[disease] += 9
            if disease in DISEASE_TO_CROP:
                crop_votes[DISEASE_TO_CROP[disease]] += 8

    crop_hint = crop_votes.most_common(1)[0][0] if crop_votes else "unknown"

    for pattern, mapping in AMBIGUOUS_DISEASE_RULES:
        if re.search(pattern, text, flags=re.IGNORECASE):
            disease = mapping.get(crop_hint, mapping["default"])
            disease_votes[disease] += 7
            if disease in DISEASE_TO_CROP:
                crop_votes[DISEASE_TO_CROP[disease]] += 8

    counterfactual_healthy = regex_any(text, [
        r"if\s+the\s+plant\s+were\s+healthy",
        r"if\s+it\s+were\s+healthy",
        r"would\s+be\s+healthy",
    ])
    if not counterfactual_healthy and regex_any(text, [
        r"\bhealthy\b", r"free\s+of\s+disease", r"disease\s+free",
        r"no\s+disease", r"unaffected", r"normal\s+leaf"
    ]):
        disease_votes["healthy"] += 2

    nonhealthy = {k: v for k, v in disease_votes.items() if k not in ["healthy", "no_leaf"]}
    if nonhealthy:
        disease = sorted(nonhealthy.items(), key=lambda kv: (-kv[1], kv[0]))[0][0]
    elif disease_votes.get("healthy", 0) > 0:
        disease = "healthy"
    else:
        disease = "unknown"

    if disease in DISEASE_TO_CROP:
        crop = DISEASE_TO_CROP[disease]
    elif crop_votes:
        crop = crop_votes.most_common(1)[0][0]
    else:
        crop = "unknown"

    if disease == "healthy":
        cls = f"{crop}_healthy" if crop != "unknown" else "unknown_healthy"
    elif disease == "unknown":
        cls = f"{crop}_unknown"
    else:
        cls = disease

    return crop, disease, cls

image_label_rows = []
for image_id, g in ann_df.groupby("image_id", sort=False):
    crop, disease, cls = infer_image_labels(g)
    image_label_rows.append({
        "image_id": image_id,
        "crop_label": crop,
        "crop_label_bn": CROP_BN.get(crop, "অজানা"),
        "disease_label": disease,
        "disease_label_bn": DISEASE_BN.get(disease, "অজানা"),
        "class_label": cls,
    })

image_label_df = pd.DataFrame(image_label_rows)
ann_df = ann_df.merge(image_label_df, on="image_id", how="left")


# ------------------------------------------------------------
# Bengali question/answer generation
# ------------------------------------------------------------
QTYPE_BN = {
    "Plant Species Identification": "ফসল প্রজাতি শনাক্তকরণ",
    "General Health Assessment": "সাধারণ স্বাস্থ্য মূল্যায়ন",
    "Specific Disease Identification": "নির্দিষ্ট রোগ শনাক্তকরণ",
    "Comprehensive Description": "বিস্তারিত বর্ণনা",
    "Causal Reasoning": "কারণভিত্তিক যুক্তি",
    "Visual Attribute Grounding": "দৃশ্যমান বৈশিষ্ট্য",
    "Detailed Verification": "যাচাইকরণ",
    "Counterfactual Reasoning": "প্রতিকল্প যুক্তি",
    "Existence": "উপস্থিতি যাচাই",
}

def stable_hash_int(text):
    return int(hashlib.md5(str(text).encode("utf-8")).hexdigest()[:8], 16)

QUESTION_TEMPLATES = {
    "Plant Species Identification": [
        "ছবির পাতাটি কোন ফসলের?",
        "এই পাতাটি কোন উদ্ভিদ প্রজাতির?",
        "পাতার ছবিতে কোন ফসল দেখা যাচ্ছে?",
    ],
    "General Health Assessment": [
        "পাতাটি কি সুস্থ?",
        "ছবির পাতাটি সুস্থ নাকি রোগাক্রান্ত?",
        "এই পাতার স্বাস্থ্য অবস্থা কী?",
    ],
    "Specific Disease Identification": [
        "এই পাতায় কোন রোগ দেখা যাচ্ছে?",
        "ছবির পাতাটির সম্ভাব্য রোগ কী?",
        "এই উদ্ভিদের পাতায় কোন রোগ শনাক্ত করা যায়?",
    ],
    "Comprehensive Description": [
        "এই পাতায় কী ধরনের দৃশ্যমান লক্ষণ দেখা যাচ্ছে?",
        "পাতার লক্ষণগুলো কীভাবে বর্ণনা করা যায়?",
        "ছবিতে পাতার রোগলক্ষণ কী দেখা যাচ্ছে?",
    ],
    "Causal Reasoning": [
        "এই রোগের সম্ভাব্য কারণ কী?",
        "পাতার এই সমস্যার কারণ কী হতে পারে?",
        "এই ধরনের লক্ষণ সাধারণত কী কারণে হয়?",
    ],
    "Visual Attribute Grounding": [
        "পাতায় কোন দৃশ্যমান বৈশিষ্ট্য দেখা যাচ্ছে?",
        "রোগ শনাক্ত করতে কোন দৃশ্যমান লক্ষণ গুরুত্বপূর্ণ?",
        "ছবিতে পাতার কোন বৈশিষ্ট্যটি বেশি স্পষ্ট?",
    ],
    "Detailed Verification": [
        "ছবির পাতাটি কি উল্লিখিত রোগের সঙ্গে মিলে?",
        "এই পাতার লক্ষণ কি নির্দিষ্ট রোগের সঙ্গে সামঞ্জস্যপূর্ণ?",
        "দৃশ্যমান লক্ষণ দেখে কি রোগটি যাচাই করা যায়?",
    ],
    "Counterfactual Reasoning": [
        "পাতাটি সুস্থ হলে কী পার্থক্য দেখা যেত?",
        "রোগ না থাকলে পাতার চেহারা কেমন হতো?",
        "সুস্থ পাতার তুলনায় এখানে কী পরিবর্তন দেখা যাচ্ছে?",
    ],
    "Existence": [
        "এই পাতায় কি রোগের কোনো লক্ষণ দেখা যাচ্ছে?",
        "ছবির পাতায় রোগের উপস্থিতি আছে কি?",
        "পাতাটিতে কি অস্বাভাবিক রোগলক্ষণ আছে?",
    ],
}

def make_bangla_question(row):
    qtype = row["question_type"]
    templates = QUESTION_TEMPLATES.get(qtype, ["ছবির পাতাটি দেখে প্রশ্নের সঠিক উত্তর কী?"])
    h = stable_hash_int(str(row["image_id"]) + str(row["question"]) + str(qtype))
    return templates[h % len(templates)]

def make_p2_answer_text(row):
    qtype = str(row["question_type"])
    ans = norm_en(row["answer"])
    crop = row["crop_label"]
    disease = row["disease_label"]
    crop_bn = CROP_BN.get(crop, "অজানা")
    disease_bn = DISEASE_BN.get(disease, "অজানা")

    if ans in {"yes", "yeah", "yep"}:
        return "হ্যাঁ।"
    if ans in {"no", "nope"}:
        return "না।"

    if disease == "no_leaf" or crop == "background":
        return "এটি একটি ব্যাকগ্রাউন্ড ছবি; এখানে গাছের পাতা নেই।"

    if qtype == "Plant Species Identification":
        if crop != "unknown":
            return f"এটি {crop_bn} গাছের পাতা।"
        return "উদ্ভিদের প্রজাতি নির্ভরযোগ্যভাবে শনাক্ত করা যায়নি।"

    if qtype == "General Health Assessment":
        if disease == "healthy":
            return "পাতাটি সুস্থ বলে মনে হচ্ছে।"
        if disease != "unknown":
            return f"পাতাটি রোগাক্রান্ত; সম্ভাব্য রোগ হলো {disease_bn}।"
        return "পাতাটির স্বাস্থ্য অবস্থা নিশ্চিতভাবে বলা যাচ্ছে না।"

    if qtype == "Specific Disease Identification":
        if disease == "healthy":
            return "পাতাটি সুস্থ; নির্দিষ্ট রোগের লক্ষণ দেখা যাচ্ছে না।"
        if disease != "unknown":
            return f"সম্ভাব্য নির্দিষ্ট রোগ হলো {disease_bn}।"
        return "নির্দিষ্ট রোগ নির্ভরযোগ্যভাবে শনাক্ত করা যায়নি।"

    if qtype == "Comprehensive Description":
        if disease == "healthy":
            return "ছবিতে পাতার সুস্থ অবস্থা দেখা যাচ্ছে; উল্লেখযোগ্য রোগের লক্ষণ নেই।"
        if disease != "unknown":
            return f"ছবিতে পাতায় {disease_bn}-এর লক্ষণ দেখা যাচ্ছে।"
        return "ছবির পাতার অবস্থা স্পষ্টভাবে নির্ধারণ করা যায়নি।"

    if qtype == "Causal Reasoning":
        if disease == "healthy":
            return "দৃশ্যমানভাবে রোগজনিত কারণের শক্ত প্রমাণ নেই।"
        if disease != "unknown":
            return f"দৃশ্যমান লক্ষণগুলো সম্ভবত {disease_bn}-এর কারণে হয়েছে।"
        return "লক্ষণগুলোর কারণ নিশ্চিতভাবে বলা যাচ্ছে না।"

    if qtype == "Visual Attribute Grounding":
        if disease == "healthy":
            return "পাতায় রোগের স্পষ্ট দাগ, ক্ষত বা অস্বাভাবিক বিবর্ণতা দেখা যাচ্ছে না।"
        if disease != "unknown":
            return f"পাতার দৃশ্যমান দাগ, বিবর্ণতা বা ক্ষত {disease_bn}-এর সঙ্গে সামঞ্জস্যপূর্ণ।"
        return "দৃশ্যমান লক্ষণ নির্ভরযোগ্যভাবে ব্যাখ্যা করা যায়নি।"

    if qtype == "Detailed Verification":
        if "yes" in ans:
            return "হ্যাঁ।"
        if "no" in ans:
            return "না।"
        if disease == "healthy":
            return "পাতাটি সুস্থ বলে মনে হচ্ছে।"
        if disease != "unknown":
            return f"হ্যাঁ, ছবির লক্ষণগুলো {disease_bn}-এর সঙ্গে সামঞ্জস্যপূর্ণ।"
        return "নির্ভরযোগ্যভাবে যাচাই করা যায়নি।"

    if qtype == "Counterfactual Reasoning":
        if disease == "healthy":
            return "যেহেতু পাতাটি সুস্থ, রোগজনিত অতিরিক্ত পরিবর্তন প্রত্যাশিত নয়।"
        if disease != "unknown":
            return f"রোগ না থাকলে {disease_bn}-এর সঙ্গে সম্পর্কিত দাগ, বিবর্ণতা বা ক্ষত অনুপস্থিত থাকত।"
        return "বিকল্প পরিস্থিতি নির্ভরযোগ্যভাবে ব্যাখ্যা করা যায়নি।"

    if disease == "healthy":
        return "পাতাটি সুস্থ বলে মনে হচ্ছে।"
    if disease != "unknown":
        return f"সম্ভাব্য রোগ হলো {disease_bn}।"
    return "উত্তর নির্ভরযোগ্যভাবে নির্ধারণ করা যায়নি।"

ann_df["question_type_bn"] = ann_df["question_type"].map(QTYPE_BN).fillna("অন্যান্য")
ann_df["bangla_question"] = ann_df.apply(make_bangla_question, axis=1)
ann_df["bangla_answer"] = ann_df.apply(make_p2_answer_text, axis=1)
ann_df["p2_answer_text"] = ann_df["bangla_answer"]
ann_df["translation_method"] = "rule_based_bn_baseline_notebook_compatible"
ann_df["needs_manual_audit"] = (
    ann_df["crop_label"].eq("unknown") |
    ann_df["disease_label"].eq("unknown") |
    ann_df["bangla_answer"].str.contains("নির্ভরযোগ্যভাবে|নিশ্চিতভাবে", regex=True)
)


# ------------------------------------------------------------
# Notebook-style leakage-safe final_split
# Source test stays test; source train gets split into train/val.
# If source split is unusable, fallback to deterministic 80/10/10 image split.
# ------------------------------------------------------------
def canonical_split(x):
    x = norm_space(x).lower()
    if "train" in x:
        return "train"
    if "test" in x:
        return "test"
    if x in {"val", "valid", "validation", "dev", "eval"} or "valid" in x:
        return "val"
    return ""

ann_df["source_split"] = ann_df["split"].map(canonical_split)

print("\nCanonical source split counts:")
print(ann_df["source_split"].value_counts(dropna=False))

image_split_df = ann_df[["image_id", "class_label", "source_split"]].drop_duplicates("image_id").copy()

has_train = (image_split_df["source_split"] == "train").sum() > 0
has_test = (image_split_df["source_split"] == "test").sum() > 0
has_val = (image_split_df["source_split"] == "val").sum() > 0

image_to_final_split = {}

if has_train and has_test:
    test_imgs = set(image_split_df.loc[image_split_df["source_split"] == "test", "image_id"].values)

    if has_val:
        val_imgs = set(image_split_df.loc[image_split_df["source_split"] == "val", "image_id"].values)
        train_imgs = set(image_split_df.loc[image_split_df["source_split"] == "train", "image_id"].values)
    else:
        train_source_images = image_split_df.loc[
            image_split_df["source_split"] == "train",
            ["image_id", "class_label"]
        ].copy()

        label_counts = train_source_images["class_label"].value_counts()
        rare_labels = set(label_counts[label_counts < 2].index)
        train_source_images["stratify_label"] = train_source_images["class_label"].apply(
            lambda x: "rare_class" if x in rare_labels else x
        )

        stratify_values = train_source_images["stratify_label"]
        can_stratify = stratify_values.value_counts().min() >= 2

        train_imgs_arr, val_imgs_arr = train_test_split(
            train_source_images["image_id"].values,
            test_size=0.10,
            random_state=SEED,
            stratify=stratify_values.values if can_stratify else None,
        )

        train_imgs = set(train_imgs_arr)
        val_imgs = set(val_imgs_arr)

    for x in train_imgs:
        image_to_final_split[x] = "train"
    for x in val_imgs:
        image_to_final_split[x] = "val"
    for x in test_imgs:
        image_to_final_split[x] = "test"

else:
    print("Source split is unusable. Falling back to deterministic image-level 80/10/10 split.")
    for image_id in image_split_df["image_id"].values:
        r = stable_hash_int(image_id) % 100
        if r < 80:
            image_to_final_split[image_id] = "train"
        elif r < 90:
            image_to_final_split[image_id] = "val"
        else:
            image_to_final_split[image_id] = "test"

ann_df["final_split"] = ann_df["image_id"].map(image_to_final_split).fillna("unused")

print("\nFinal row split counts:")
print(ann_df["final_split"].value_counts(dropna=False))

print("\nFinal image split counts:")
print(ann_df[["image_id", "final_split"]].drop_duplicates()["final_split"].value_counts(dropna=False))

ann_df = ann_df[ann_df["final_split"].isin(["train", "val", "test"])].reset_index(drop=True)

if len(ann_df) == 0:
    raise RuntimeError("No rows left after final_split assignment.")

# Leakage check
split_sets = {
    s: set(ann_df.loc[ann_df["final_split"] == s, "image_id"].unique())
    for s in ["train", "val", "test"]
}

assert len(split_sets["train"] & split_sets["val"]) == 0, "train/val leakage"
assert len(split_sets["train"] & split_sets["test"]) == 0, "train/test leakage"
assert len(split_sets["val"] & split_sets["test"]) == 0, "val/test leakage"

if len(split_sets["train"]) == 0 or len(split_sets["val"]) == 0 or len(split_sets["test"]) == 0:
    raise RuntimeError(
        f"Bad image split sizes after robust split: "
        f"train={len(split_sets['train'])}, val={len(split_sets['val'])}, test={len(split_sets['test'])}"
    )

print("Leakage-safe split passed.")


# ------------------------------------------------------------
# QA id
# ------------------------------------------------------------
ann_df["qa_id"] = [
    f"{sp}_{i:06d}_{hashlib.md5((str(img) + str(q) + str(a)).encode('utf-8')).hexdigest()[:8]}"
    for i, (sp, img, q, a) in enumerate(zip(
        ann_df["final_split"], ann_df["image_id"], ann_df["question"], ann_df["answer"]
    ))
]


# ------------------------------------------------------------
# P2 ids and mappings: train vocabulary only, like notebook
# ------------------------------------------------------------
train_answer_counts = ann_df.loc[ann_df["final_split"] == "train", "p2_answer_text"].value_counts()
p2_answer_vocab = train_answer_counts.index.tolist()

UNK_ANSWER = "<unk_answer>"
if UNK_ANSWER not in p2_answer_vocab:
    p2_answer_vocab.append(UNK_ANSWER)

p2_answer_to_id = {a: i for i, a in enumerate(p2_answer_vocab)}
ann_df["p2_answer_id"] = (
    ann_df["p2_answer_text"]
    .map(p2_answer_to_id)
    .fillna(p2_answer_to_id[UNK_ANSWER])
    .astype(int)
)

p2_crop_vocab = sorted(ann_df["crop_label"].dropna().unique().tolist())
p2_disease_vocab = sorted(ann_df["disease_label"].dropna().unique().tolist())
p2_class_vocab = sorted(ann_df["class_label"].dropna().unique().tolist())

p2_crop_to_id = {x: i for i, x in enumerate(p2_crop_vocab)}
p2_disease_to_id = {x: i for i, x in enumerate(p2_disease_vocab)}
p2_class_to_id = {x: i for i, x in enumerate(p2_class_vocab)}

ann_df["p2_crop_label"] = ann_df["crop_label"]
ann_df["p2_disease_label"] = ann_df["disease_label"]
ann_df["p2_class_label"] = ann_df["class_label"]

ann_df["p2_crop_id"] = ann_df["p2_crop_label"].map(p2_crop_to_id).astype(int)
ann_df["p2_disease_id"] = ann_df["p2_disease_label"].map(p2_disease_to_id).astype(int)
ann_df["p2_class_id"] = ann_df["p2_class_label"].map(p2_class_to_id).astype(int)

ann_df["p2_crop_loss_mask"] = ann_df["p2_crop_label"].ne("unknown").astype(int)
ann_df["p2_disease_loss_mask"] = ann_df["p2_disease_label"].ne("unknown").astype(int)
ann_df["p2_class_loss_mask"] = ann_df["p2_class_label"].ne("unknown_unknown").astype(int)
ann_df["p2_label_source"] = "row_level_rule_baseline_compatible"


# ------------------------------------------------------------
# Save P1-style processed files
# ------------------------------------------------------------
p1_cols = [
    "qa_id",
    "image_id",
    "resolved_image_path",
    "final_split",
    "question_type",
    "question_type_bn",
    "question",
    "answer",
    "bangla_question",
    "bangla_answer",
    "crop_label",
    "crop_label_bn",
    "disease_label",
    "disease_label_bn",
    "class_label",
    "translation_method",
    "needs_manual_audit",
]

final_df = ann_df[p1_cols].copy()

final_df.to_csv(PROCESSED_DIR / "plantvillagevqa_bn_clean.csv", index=False)
final_df[final_df["final_split"] == "train"].to_csv(PROCESSED_DIR / "train_bn.csv", index=False)
final_df[final_df["final_split"] == "val"].to_csv(PROCESSED_DIR / "val_bn.csv", index=False)
final_df[final_df["final_split"] == "test"].to_csv(PROCESSED_DIR / "test_bn.csv", index=False)

with open(PROCESSED_DIR / "crop_name_bn_dictionary.json", "w", encoding="utf-8") as f:
    json.dump(CROP_BN, f, ensure_ascii=False, indent=2)

with open(PROCESSED_DIR / "disease_name_bn_dictionary.json", "w", encoding="utf-8") as f:
    json.dump(DISEASE_BN, f, ensure_ascii=False, indent=2)


# ------------------------------------------------------------
# Save P2 model-ready files
# ------------------------------------------------------------
p2_keep_cols = [
    "qa_id",
    "image_id",
    "resolved_image_path",
    "final_split",
    "question_type",
    "question_type_bn",
    "question",
    "answer",
    "bangla_question",
    "bangla_answer",
    "p2_answer_text",
    "p2_answer_id",
    "p2_crop_label",
    "p2_crop_id",
    "p2_crop_loss_mask",
    "p2_disease_label",
    "p2_disease_id",
    "p2_disease_loss_mask",
    "p2_class_label",
    "p2_class_id",
    "p2_class_loss_mask",
    "p2_label_source",
]

p2_model_df = ann_df[p2_keep_cols].copy()

p2_train_df = p2_model_df[p2_model_df["final_split"] == "train"].reset_index(drop=True)
p2_val_df = p2_model_df[p2_model_df["final_split"] == "val"].reset_index(drop=True)
p2_test_df = p2_model_df[p2_model_df["final_split"] == "test"].reset_index(drop=True)

if len(p2_train_df) == 0 or len(p2_val_df) == 0 or len(p2_test_df) == 0:
    raise RuntimeError(
        f"Bad split sizes after P2 build: "
        f"train={len(p2_train_df)}, val={len(p2_val_df)}, test={len(p2_test_df)}"
    )

p2_model_df.to_csv(P2_FULL_CSV, index=False)
p2_train_df.to_csv(P2_TRAIN_CSV, index=False)
p2_val_df.to_csv(P2_VAL_CSV, index=False)
p2_test_df.to_csv(P2_TEST_CSV, index=False)

label_mapping = {
    "answer_to_id": p2_answer_to_id,
    "crop_to_id": p2_crop_to_id,
    "disease_to_id": p2_disease_to_id,
    "class_to_id": p2_class_to_id,
    "id_to_answer": {str(v): k for k, v in p2_answer_to_id.items()},
    "id_to_crop": {str(v): k for k, v in p2_crop_to_id.items()},
    "id_to_disease": {str(v): k for k, v in p2_disease_to_id.items()},
    "id_to_class": {str(v): k for k, v in p2_class_to_id.items()},
}

with open(P2_MAPPING_JSON, "w", encoding="utf-8") as f:
    json.dump(label_mapping, f, ensure_ascii=False, indent=2)


# ------------------------------------------------------------
# Build Bengali tokenizer, same format as B2 expects
# ------------------------------------------------------------
TOKEN_RE = re.compile(r"[\u0980-\u09FF]+|[A-Za-z0-9]+|[^\s]", flags=re.UNICODE)

def tokenize_bn(text):
    text = "" if pd.isna(text) else str(text)
    return TOKEN_RE.findall(text.strip())

max_vocab_size = 12000
min_freq = 1
max_len = 48

special_tokens = ["<pad>", "<unk>", "<cls>", "<sep>"]
counter = Counter()

for q in p2_train_df["bangla_question"].fillna("").astype(str).tolist():
    counter.update(tokenize_bn(q))

valid_tokens = [
    tok for tok, freq in sorted(counter.items(), key=lambda kv: (-kv[1], kv[0]))
    if freq >= min_freq and tok not in special_tokens
]

valid_tokens = valid_tokens[: max_vocab_size - len(special_tokens)]

token_to_id = {tok: i for i, tok in enumerate(special_tokens)}
for tok in valid_tokens:
    token_to_id[tok] = len(token_to_id)

tokenizer_obj = {
    "max_vocab_size": max_vocab_size,
    "min_freq": min_freq,
    "max_len": max_len,
    "token_to_id": token_to_id,
}

with open(P2_TOKENIZER_JSON, "w", encoding="utf-8") as f:
    json.dump(tokenizer_obj, f, ensure_ascii=False, indent=2)


# ------------------------------------------------------------
# Reload variables expected by later baseline cells
# ------------------------------------------------------------
train_df = pd.read_csv(P2_TRAIN_CSV)
val_df = pd.read_csv(P2_VAL_CSV)
test_df = pd.read_csv(P2_TEST_CSV)

with open(P2_MAPPING_JSON, "r", encoding="utf-8") as f:
    label_mapping = json.load(f)

num_answers = len(label_mapping["answer_to_id"])
num_crops = len(label_mapping["crop_to_id"])
num_diseases = len(label_mapping["disease_to_id"])
num_classes = len(label_mapping["class_to_id"])

# Final leakage guard
train_ids = set(train_df["image_id"].unique())
val_ids = set(val_df["image_id"].unique())
test_ids = set(test_df["image_id"].unique())

assert len(train_ids & val_ids) == 0, "train/val image leakage"
assert len(train_ids & test_ids) == 0, "train/test image leakage"
assert len(val_ids & test_ids) == 0, "val/test image leakage"

print("\n" + "=" * 90)
print("B1-R2 COMPLETE")
print("=" * 90)

print("Rows:")
print("train:", len(train_df), "| images:", train_df["image_id"].nunique())
print("val:", len(val_df), "| images:", val_df["image_id"].nunique())
print("test:", len(test_df), "| images:", test_df["image_id"].nunique())

print("\nLabel sizes:")
print("answers:", num_answers)
print("crops:", num_crops)
print("diseases:", num_diseases)
print("classes:", num_classes)

print("\nTokenizer vocab size:", len(token_to_id))
print("Tokenizer max_len:", max_len)

print("\nSaved P2 artifacts:")
print(P2_TRAIN_CSV)
print(P2_VAL_CSV)
print(P2_TEST_CSV)
print(P2_MAPPING_JSON)
print(P2_TOKENIZER_JSON)

print("\nSample:")
display(train_df[[
    "qa_id",
    "image_id",
    "final_split",
    "question_type",
    "bangla_question",
    "p2_answer_text",
    "p2_answer_id",
    "resolved_image_path",
]].head())

gc.collect()

B1-R2: BUILDING P2 ARTIFACTS LIKE THE GIVEN NOTEBOOK
Searching/downloading Hugging Face dataset: SyedNazmusSakib/PlantVillageVQA


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

ZIP path: /kaggle/working/tinyagrivqa_bn/cache/hf_snapshot/PlantVillageVQA.zip
ZIP size GB: 0.811
Extracting ZIP to: /kaggle/working/tinyagrivqa_bn/PlantVillageVQA_extracted
Annotation CSV: /kaggle/working/tinyagrivqa_bn/PlantVillageVQA_extracted/PlantVillageVQA/PlantVillageVQA.csv
Raw annotation shape: (193609, 6)
Raw split counts:
split
train    154977
test      38632
Name: count, dtype: int64

Resolving image paths by image_id basename...
Image files found: 55448
Unique image basenames: 55448
Missing image rows after basename resolution: 0
Rows after image resolution: 193609
Unique resolved images: 50073

Canonical source split counts:
source_split
train    154977
test      38632
Name: count, dtype: int64

Final row split counts:
final_split
train    139455
test      38632
val       15522
Name: count, dtype: int64

Final image split counts:
final_split
train    36059
test     10007
val       4007
Name: count, dtype: int64
Leakage-safe split passed.

B1-R2 COMPLETE
Rows:
train: 13945

,qa_id,image_id,final_split,question_type,bangla_question,p2_answer_text,p2_answer_id,resolved_image_path
0,train_000000_8a79e2d6,image_000001.jpg,train,Existence & Sanity Check,ছবির পাতাটি দেখে প্রশ্নের সঠিক উত্তর কী?,না।,0,/kaggle/working/tinyagrivqa_bn/PlantVillageVQA...
1,train_000001_e681bbf1,image_000001.jpg,train,Existence & Sanity Check,ছবির পাতাটি দেখে প্রশ্নের সঠিক উত্তর কী?,না।,0,/kaggle/working/tinyagrivqa_bn/PlantVillageVQA...
2,train_000002_f358910f,image_000001.jpg,train,Existence & Sanity Check,ছবির পাতাটি দেখে প্রশ্নের সঠিক উত্তর কী?,এটি একটি ব্যাকগ্রাউন্ড ছবি; এখানে গাছের পাতা নেই।,40,/kaggle/working/tinyagrivqa_bn/PlantVillageVQA...
3,train_000004_ecf30569,image_000003.jpg,train,Specific Disease Identification,এই উদ্ভিদের পাতায় কোন রোগ শনাক্ত করা যায়?,পাতাটি সুস্থ; নির্দিষ্ট রোগের লক্ষণ দেখা যাচ্ছ...,12,/kaggle/working/tinyagrivqa_bn/PlantVillageVQA...
4,train_000005_da64adfa,image_000004.JPG,train,Comprehensive Description,ছবিতে পাতার রোগলক্ষণ কী দেখা যাচ্ছে?,ছবিতে পাতার সুস্থ অবস্থা দেখা যাচ্ছে; উল্লেখযো...,19,/kaggle/working/tinyagrivqa_bn/PlantVillageVQA...


20

In [3]:
# ============================================================
# Baseline Portion B2: Tokenizer, transforms, dataset
# ============================================================

print("=" * 90)
print("TOKENIZER / DATASET")
print("=" * 90)

TOKEN_RE = re.compile(r"[\u0980-\u09FF]+|[A-Za-z0-9]+|[^\s]", flags=re.UNICODE)


class BengaliWordTokenizer:
    def __init__(self, max_vocab_size=12000, min_freq=1, max_len=48):
        self.max_vocab_size = max_vocab_size
        self.min_freq = min_freq
        self.max_len = max_len

        self.pad_token = "<pad>"
        self.unk_token = "<unk>"
        self.cls_token = "<cls>"
        self.sep_token = "<sep>"

        self.token_to_id = {
            self.pad_token: 0,
            self.unk_token: 1,
            self.cls_token: 2,
            self.sep_token: 3,
        }
        self.id_to_token = {v: k for k, v in self.token_to_id.items()}

    def tokenize(self, text):
        text = "" if pd.isna(text) else str(text)
        return TOKEN_RE.findall(text.strip())

    def encode(self, text):
        tokens = [self.cls_token] + self.tokenize(text)[: self.max_len - 2] + [self.sep_token]
        ids = [self.token_to_id.get(tok, self.token_to_id[self.unk_token]) for tok in tokens]

        attention_mask = [1] * len(ids)

        pad_len = self.max_len - len(ids)
        if pad_len > 0:
            ids += [self.token_to_id[self.pad_token]] * pad_len
            attention_mask += [0] * pad_len

        return np.array(ids, dtype=np.int64), np.array(attention_mask, dtype=np.int64)

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            obj = json.load(f)

        tok = cls(
            max_vocab_size=obj["max_vocab_size"],
            min_freq=obj["min_freq"],
            max_len=obj["max_len"],
        )

        tok.token_to_id = {k: int(v) for k, v in obj["token_to_id"].items()}
        tok.id_to_token = {int(v): k for k, v in tok.token_to_id.items()}
        return tok


tokenizer = BengaliWordTokenizer.load(P2_TOKENIZER_JSON)

print("Tokenizer vocab size:", len(tokenizer.token_to_id))
print("Tokenizer max_len:", tokenizer.max_len)


# ------------------------------------------------------------
# Image transforms
# ------------------------------------------------------------
try:
    import torchvision
    from torchvision import transforms
    TORCHVISION_OK = True
    print("torchvision:", torchvision.__version__)
except Exception as e:
    TORCHVISION_OK = False
    print("torchvision unavailable:", repr(e))


class SimpleImageTransform:
    def __init__(self, image_size=224):
        self.image_size = image_size
        self.mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        self.std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    def __call__(self, img):
        img = img.resize((self.image_size, self.image_size))
        arr = np.array(img).astype(np.float32) / 255.0
        x = torch.from_numpy(arr).permute(2, 0, 1)  # [3, H, W]
        x = (x - self.mean) / self.std
        return x


def build_image_transform(image_size=224, train=False):
    if TORCHVISION_OK:
        if train:
            return transforms.Compose([
                transforms.Resize((image_size, image_size)),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomApply([
                    transforms.ColorJitter(
                        brightness=0.15,
                        contrast=0.15,
                        saturation=0.10,
                        hue=0.02,
                    )
                ], p=0.25),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225],
                ),
            ])

        return transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
            ),
        ])

    return SimpleImageTransform(image_size=image_size)


# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------
class TinyAgriVQABaselineDataset(Dataset):
    def __init__(self, df, tokenizer, image_transform=None, split_name="train"):
        self.df = df.reset_index(drop=True).copy()
        self.tokenizer = tokenizer
        self.image_transform = image_transform
        self.split_name = split_name

        self.image_paths = self.df["resolved_image_path"].astype(str).tolist()
        self.qa_ids = self.df["qa_id"].astype(str).tolist()
        self.image_ids = self.df["image_id"].astype(str).tolist()

        questions = self.df["bangla_question"].fillna("").astype(str).tolist()

        ids_list = []
        mask_list = []

        for q in tqdm(questions, desc=f"pre-encode {split_name} questions"):
            ids, mask = tokenizer.encode(q)
            ids_list.append(ids)
            mask_list.append(mask)

        self.input_ids = np.stack(ids_list, axis=0).astype(np.int64)
        self.attention_masks = np.stack(mask_list, axis=0).astype(np.int64)
        self.answer_ids = self.df["p2_answer_id"].astype(np.int64).values

    def __len__(self):
        return len(self.df)

    def _load_image(self, path):
        try:
            with Image.open(path) as img:
                return img.convert("RGB")
        except Exception:
            return Image.new("RGB", (224, 224), color=(0, 0, 0))

    def __getitem__(self, idx):
        img = self._load_image(self.image_paths[idx])

        if self.image_transform is not None:
            img = self.image_transform(img)

        return {
            "image": img.float(),                                                   # [3, H, W]
            "input_ids": torch.from_numpy(self.input_ids[idx]).long(),               # [L]
            "attention_mask": torch.from_numpy(self.attention_masks[idx]).long(),    # [L]
            "answer_id": torch.tensor(self.answer_ids[idx], dtype=torch.long),       # []
            "qa_id": self.qa_ids[idx],
            "image_id": self.image_ids[idx],
        }


train_transform = build_image_transform(BASE_CFG.image_size, train=True)
eval_transform = build_image_transform(BASE_CFG.image_size, train=False)

train_ds = TinyAgriVQABaselineDataset(train_df, tokenizer, train_transform, split_name="train")
val_ds = TinyAgriVQABaselineDataset(val_df, tokenizer, eval_transform, split_name="val")
test_ds = TinyAgriVQABaselineDataset(test_df, tokenizer, eval_transform, split_name="test")

sample = train_ds[0]

print("Datasets:")
print("train:", len(train_ds))
print("val:", len(val_ds))
print("test:", len(test_ds))

print("\nSample shapes:")
print("image:", tuple(sample["image"].shape))
print("input_ids:", tuple(sample["input_ids"].shape))
print("attention_mask:", tuple(sample["attention_mask"].shape))
print("answer_id:", sample["answer_id"])

TOKENIZER / DATASET
Tokenizer vocab size: 75
Tokenizer max_len: 48
torchvision: 0.25.0+cu128


pre-encode train questions:   0%|          | 0/139455 [00:00<?, ?it/s]

pre-encode val questions:   0%|          | 0/15522 [00:00<?, ?it/s]

pre-encode test questions:   0%|          | 0/38632 [00:00<?, ?it/s]

Datasets:
train: 139455
val: 15522
test: 38632

Sample shapes:
image: (3, 224, 224)
input_ids: (48,)
attention_mask: (48,)
answer_id: tensor(0)


In [4]:
# ============================================================
# Baseline Portion B2: Tokenizer, transforms, dataset
# ============================================================

print("=" * 90)
print("TOKENIZER / DATASET")
print("=" * 90)

TOKEN_RE = re.compile(r"[\u0980-\u09FF]+|[A-Za-z0-9]+|[^\s]", flags=re.UNICODE)


class BengaliWordTokenizer:
    def __init__(self, max_vocab_size=12000, min_freq=1, max_len=48):
        self.max_vocab_size = max_vocab_size
        self.min_freq = min_freq
        self.max_len = max_len

        self.pad_token = "<pad>"
        self.unk_token = "<unk>"
        self.cls_token = "<cls>"
        self.sep_token = "<sep>"

        self.token_to_id = {
            self.pad_token: 0,
            self.unk_token: 1,
            self.cls_token: 2,
            self.sep_token: 3,
        }
        self.id_to_token = {v: k for k, v in self.token_to_id.items()}

    def tokenize(self, text):
        text = "" if pd.isna(text) else str(text)
        return TOKEN_RE.findall(text.strip())

    def encode(self, text):
        tokens = [self.cls_token] + self.tokenize(text)[: self.max_len - 2] + [self.sep_token]
        ids = [self.token_to_id.get(tok, self.token_to_id[self.unk_token]) for tok in tokens]

        attention_mask = [1] * len(ids)

        pad_len = self.max_len - len(ids)
        if pad_len > 0:
            ids += [self.token_to_id[self.pad_token]] * pad_len
            attention_mask += [0] * pad_len

        return np.array(ids, dtype=np.int64), np.array(attention_mask, dtype=np.int64)

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            obj = json.load(f)

        tok = cls(
            max_vocab_size=obj["max_vocab_size"],
            min_freq=obj["min_freq"],
            max_len=obj["max_len"],
        )

        tok.token_to_id = {k: int(v) for k, v in obj["token_to_id"].items()}
        tok.id_to_token = {int(v): k for k, v in tok.token_to_id.items()}
        return tok


tokenizer = BengaliWordTokenizer.load(P2_TOKENIZER_JSON)

print("Tokenizer vocab size:", len(tokenizer.token_to_id))
print("Tokenizer max_len:", tokenizer.max_len)


# ------------------------------------------------------------
# Image transforms
# ------------------------------------------------------------
try:
    import torchvision
    from torchvision import transforms
    TORCHVISION_OK = True
    print("torchvision:", torchvision.__version__)
except Exception as e:
    TORCHVISION_OK = False
    print("torchvision unavailable:", repr(e))


class SimpleImageTransform:
    def __init__(self, image_size=224):
        self.image_size = image_size
        self.mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        self.std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    def __call__(self, img):
        img = img.resize((self.image_size, self.image_size))
        arr = np.array(img).astype(np.float32) / 255.0
        x = torch.from_numpy(arr).permute(2, 0, 1)  # [3, H, W]
        x = (x - self.mean) / self.std
        return x


def build_image_transform(image_size=224, train=False):
    if TORCHVISION_OK:
        if train:
            return transforms.Compose([
                transforms.Resize((image_size, image_size)),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomApply([
                    transforms.ColorJitter(
                        brightness=0.15,
                        contrast=0.15,
                        saturation=0.10,
                        hue=0.02,
                    )
                ], p=0.25),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225],
                ),
            ])

        return transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
            ),
        ])

    return SimpleImageTransform(image_size=image_size)


# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------
class TinyAgriVQABaselineDataset(Dataset):
    def __init__(self, df, tokenizer, image_transform=None, split_name="train"):
        self.df = df.reset_index(drop=True).copy()
        self.tokenizer = tokenizer
        self.image_transform = image_transform
        self.split_name = split_name

        self.image_paths = self.df["resolved_image_path"].astype(str).tolist()
        self.qa_ids = self.df["qa_id"].astype(str).tolist()
        self.image_ids = self.df["image_id"].astype(str).tolist()

        questions = self.df["bangla_question"].fillna("").astype(str).tolist()

        ids_list = []
        mask_list = []

        for q in tqdm(questions, desc=f"pre-encode {split_name} questions"):
            ids, mask = tokenizer.encode(q)
            ids_list.append(ids)
            mask_list.append(mask)

        self.input_ids = np.stack(ids_list, axis=0).astype(np.int64)
        self.attention_masks = np.stack(mask_list, axis=0).astype(np.int64)
        self.answer_ids = self.df["p2_answer_id"].astype(np.int64).values

    def __len__(self):
        return len(self.df)

    def _load_image(self, path):
        try:
            with Image.open(path) as img:
                return img.convert("RGB")
        except Exception:
            return Image.new("RGB", (224, 224), color=(0, 0, 0))

    def __getitem__(self, idx):
        img = self._load_image(self.image_paths[idx])

        if self.image_transform is not None:
            img = self.image_transform(img)

        return {
            "image": img.float(),                                                   # [3, H, W]
            "input_ids": torch.from_numpy(self.input_ids[idx]).long(),               # [L]
            "attention_mask": torch.from_numpy(self.attention_masks[idx]).long(),    # [L]
            "answer_id": torch.tensor(self.answer_ids[idx], dtype=torch.long),       # []
            "qa_id": self.qa_ids[idx],
            "image_id": self.image_ids[idx],
        }


train_transform = build_image_transform(BASE_CFG.image_size, train=True)
eval_transform = build_image_transform(BASE_CFG.image_size, train=False)

train_ds = TinyAgriVQABaselineDataset(train_df, tokenizer, train_transform, split_name="train")
val_ds = TinyAgriVQABaselineDataset(val_df, tokenizer, eval_transform, split_name="val")
test_ds = TinyAgriVQABaselineDataset(test_df, tokenizer, eval_transform, split_name="test")

sample = train_ds[0]

print("Datasets:")
print("train:", len(train_ds))
print("val:", len(val_ds))
print("test:", len(test_ds))

print("\nSample shapes:")
print("image:", tuple(sample["image"].shape))
print("input_ids:", tuple(sample["input_ids"].shape))
print("attention_mask:", tuple(sample["attention_mask"].shape))
print("answer_id:", sample["answer_id"])

TOKENIZER / DATASET
Tokenizer vocab size: 75
Tokenizer max_len: 48
torchvision: 0.25.0+cu128


pre-encode train questions:   0%|          | 0/139455 [00:00<?, ?it/s]

pre-encode val questions:   0%|          | 0/15522 [00:00<?, ?it/s]

pre-encode test questions:   0%|          | 0/38632 [00:00<?, ?it/s]

Datasets:
train: 139455
val: 15522
test: 38632

Sample shapes:
image: (3, 224, 224)
input_ids: (48,)
attention_mask: (48,)
answer_id: tensor(0)


In [5]:
# ============================================================
# Baseline Portion B3: Model architecture
# ============================================================

print("=" * 90)
print("MODEL: ResNet18 + Bengali Transformer + Concat Fusion")
print("=" * 90)


class FallbackTinyCNN(nn.Module):
    def __init__(self, out_dim=512):
        super().__init__()
        self.out_dim = out_dim

        self.net = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1),    # [B, 32, 112, 112]
            nn.BatchNorm2d(32),
            nn.SiLU(),

            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),   # [B, 64, 56, 56]
            nn.BatchNorm2d(64),
            nn.SiLU(),

            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),  # [B, 128, 28, 28]
            nn.BatchNorm2d(128),
            nn.SiLU(),

            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1), # [B, 256, 14, 14]
            nn.BatchNorm2d(256),
            nn.SiLU(),

            nn.Conv2d(256, out_dim, kernel_size=3, stride=2, padding=1), # [B, D, 7, 7]
            nn.BatchNorm2d(out_dim),
            nn.SiLU(),

            nn.AdaptiveAvgPool2d(1), # [B, D, 1, 1]
        )

    def forward(self, x):
        x = self.net(x)       # [B, D, 1, 1]
        x = x.flatten(1)      # [B, D]
        return x


class ResNet18ImageEncoder(nn.Module):
    def __init__(self, pretrained=False):
        super().__init__()

        self.pretrained = pretrained

        if TORCHVISION_OK:
            from torchvision import models

            weights = None
            if pretrained:
                try:
                    weights = models.ResNet18_Weights.DEFAULT
                except Exception:
                    weights = None

            model = models.resnet18(weights=weights)
            self.out_dim = model.fc.in_features
            model.fc = nn.Identity()
            self.encoder = model
        else:
            self.out_dim = 512
            self.encoder = FallbackTinyCNN(out_dim=self.out_dim)

    def forward(self, images):
        # images: [B, 3, H, W]
        feats = self.encoder(images)  # [B, 512]
        return feats


class TinyTextTransformer(nn.Module):
    def __init__(
        self,
        vocab_size,
        max_len=48,
        d_model=256,
        n_heads=4,
        n_layers=3,
        ff_dim=768,
        dropout=0.10,
        pad_id=0,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.max_len = max_len
        self.d_model = d_model
        self.pad_id = pad_id

        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos_emb = nn.Parameter(torch.zeros(1, max_len, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer=encoder_layer,
            num_layers=n_layers,
        )

        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

        nn.init.normal_(self.pos_emb, mean=0.0, std=0.02)

    def forward(self, input_ids, attention_mask):
        # input_ids: [B, L]
        # attention_mask: [B, L]
        B, L = input_ids.shape

        x = self.token_emb(input_ids)                  # [B, L, D]
        x = x + self.pos_emb[:, :L, :]                 # [B, L, D]
        x = self.dropout(x)

        key_padding_mask = attention_mask.eq(0)        # [B, L]

        h = self.encoder(
            x,
            src_key_padding_mask=key_padding_mask,
        )                                             # [B, L, D]

        h = self.norm(h)                              # [B, L, D]

        mask = attention_mask.unsqueeze(-1).float()    # [B, L, 1]
        pooled = (h * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1.0) # [B, D]

        return pooled, h


class ConcatFusion(nn.Module):
    def __init__(self, image_dim=512, text_dim=256, fusion_dim=512, dropout=0.10):
        super().__init__()

        self.image_proj = nn.Sequential(
            nn.Linear(image_dim, fusion_dim),
            nn.LayerNorm(fusion_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.text_proj = nn.Sequential(
            nn.Linear(text_dim, fusion_dim),
            nn.LayerNorm(fusion_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.fusion = nn.Sequential(
            nn.Linear(fusion_dim * 2, fusion_dim),
            nn.LayerNorm(fusion_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, image_feat, text_feat):
        # image_feat: [B, image_dim]
        # text_feat: [B, text_dim]

        v = self.image_proj(image_feat)       # [B, F]
        t = self.text_proj(text_feat)         # [B, F]

        z = torch.cat([v, t], dim=-1)         # [B, 2F]
        fused = self.fusion(z)                # [B, F]

        return fused


class ResNet18TransformerConcatVQA(nn.Module):
    def __init__(
        self,
        vocab_size,
        num_answers,
        max_len=48,
        image_pretrained=False,
        text_dim=256,
        text_layers=3,
        text_heads=4,
        text_ff_dim=768,
        fusion_dim=512,
        dropout=0.10,
        pad_id=0,
    ):
        super().__init__()

        self.image_encoder = ResNet18ImageEncoder(
            pretrained=image_pretrained,
        )

        self.text_encoder = TinyTextTransformer(
            vocab_size=vocab_size,
            max_len=max_len,
            d_model=text_dim,
            n_heads=text_heads,
            n_layers=text_layers,
            ff_dim=text_ff_dim,
            dropout=dropout,
            pad_id=pad_id,
        )

        self.fusion = ConcatFusion(
            image_dim=self.image_encoder.out_dim,
            text_dim=text_dim,
            fusion_dim=fusion_dim,
            dropout=dropout,
        )

        self.answer_head = nn.Sequential(
            nn.Linear(fusion_dim, fusion_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_dim, num_answers),
        )

    def forward(self, images, input_ids, attention_mask):
        # images: [B, 3, H, W]
        # input_ids: [B, L]
        # attention_mask: [B, L]

        image_feat = self.image_encoder(images)                         # [B, 512]
        text_feat, token_features = self.text_encoder(input_ids, attention_mask) # [B, text_dim], [B, L, text_dim]

        fused = self.fusion(image_feat, text_feat)                      # [B, fusion_dim]

        answer_logits = self.answer_head(fused)                         # [B, num_answers]

        return {
            "answer_logits": answer_logits,
            "fused_features": fused,
            "image_features": image_feat,
            "text_features": text_feat,
            "token_features": token_features,
        }


def build_baseline_model(cfg: BaselineConfig):
    return ResNet18TransformerConcatVQA(
        vocab_size=len(tokenizer.token_to_id),
        num_answers=num_answers,
        max_len=cfg.max_text_len,
        image_pretrained=cfg.image_pretrained,
        text_dim=cfg.text_dim,
        text_layers=cfg.text_layers,
        text_heads=cfg.text_heads,
        text_ff_dim=cfg.text_ff_dim,
        fusion_dim=cfg.fusion_dim,
        dropout=cfg.dropout,
        pad_id=tokenizer.token_to_id[tokenizer.pad_token],
    )


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

    return {
        "total": int(total),
        "trainable": int(trainable),
        "total_m": total / 1e6,
        "trainable_m": trainable / 1e6,
    }


tmp_model = build_baseline_model(BASE_CFG)
param_report = count_parameters(tmp_model)

print("Parameter report:")
print(json.dumps(param_report, indent=2))

del tmp_model
gc.collect()

MODEL: ResNet18 + Bengali Transformer + Concat Fusion
Parameter report:
{
  "total": 14432701,
  "trainable": 14432701,
  "total_m": 14.432701,
  "trainable_m": 14.432701
}


89

In [6]:
# ============================================================
# Baseline Portion B5: Eval and checkpoint utilities
# ============================================================

print("=" * 90)
print("EVAL / CHECKPOINT UTILS")
print("=" * 90)


def save_baseline_checkpoint(path, model, optimizer, scheduler, epoch, best_metric, history, cfg):
    model_to_save = unwrap_model(model)

    torch.save(
        {
            "epoch": int(epoch),
            "model_state_dict": model_to_save.state_dict(),
            "optimizer_state_dict": optimizer.state_dict() if optimizer is not None else None,
            "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
            "best_metric": float(best_metric),
            "history": history,
            "model_config": {
                "model_name": "ResNet18TransformerConcatVQA",
                "image_backbone": "resnet18",
                "fusion": "concat",
                "vocab_size": len(tokenizer.token_to_id),
                "num_answers": num_answers,
                "max_len": cfg.max_text_len,
                "image_pretrained": cfg.image_pretrained,
                "text_dim": cfg.text_dim,
                "text_layers": cfg.text_layers,
                "text_heads": cfg.text_heads,
                "text_ff_dim": cfg.text_ff_dim,
                "fusion_dim": cfg.fusion_dim,
                "dropout": cfg.dropout,
                "pad_id": tokenizer.token_to_id[tokenizer.pad_token],
            },
            "training_config": asdict(cfg),
            "label_mapping": label_mapping,
            "tokenizer_path": str(P2_TOKENIZER_JSON),
        },
        path,
    )


def evaluate_baseline_model(model, loader, device, cfg: BaselineConfig, max_batches=None):
    model.eval()

    all_logits = []
    all_labels = []

    loss_meter = defaultdict(float)
    n_batches = 0

    with torch.no_grad():
        pbar = tqdm(loader, desc="eval", leave=False)

        for step, batch in enumerate(pbar):
            if max_batches is not None and step >= max_batches:
                break

            batch = batch_to_device(batch, device)

            outputs = model(
                images=batch["image"],
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
            )

            loss_dict = compute_baseline_loss(outputs, batch, cfg)

            for k, v in loss_dict.items():
                loss_meter[k] += float(v.detach().cpu())

            all_logits.append(outputs["answer_logits"].detach().cpu())
            all_labels.append(batch["answer_id"].detach().cpu())

            n_batches += 1

    logits_np = torch.cat(all_logits, dim=0).numpy()
    labels_np = torch.cat(all_labels, dim=0).numpy()

    metrics = compute_answer_metrics(logits_np, labels_np)

    for k in list(loss_meter.keys()):
        metrics[k] = loss_meter[k] / max(1, n_batches)

    return metrics


print("Eval/checkpoint utilities ready.")

EVAL / CHECKPOINT UTILS
Eval/checkpoint utilities ready.


In [7]:
# ============================================================
# Baseline Portion B6: Trainer
# ============================================================

print("=" * 90)
print("BASELINE TRAINER")
print("=" * 90)


def train_resnet18_concat_baseline(cfg: BaselineConfig):
    seed_everything(BASE_SEED)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_loader, val_loader, _ = build_dataloaders(cfg)

    model = build_baseline_model(cfg)
    param_report = count_parameters(model)

    model = model.to(device)

    if torch.cuda.device_count() > 1:
        print("Multiple GPUs detected. Using nn.DataParallel.")
        model = nn.DataParallel(model)

    optimizer = build_optimizer(model, cfg)

    steps_per_epoch = math.ceil(len(train_loader) / cfg.gradient_accumulation_steps)
    total_steps = steps_per_epoch * cfg.epochs
    scheduler = build_cosine_scheduler(optimizer, total_steps, cfg.warmup_ratio)

    use_amp = bool(device.type == "cuda" and cfg.mixed_precision == "fp16")

    try:
        scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    except Exception:
        scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    best_metric = -1.0
    best_epoch = -1
    bad_epochs = 0
    history = []

    best_ckpt_path = BASE_CKPT_DIR / "best_resnet18_transformer_concat.pt"
    last_ckpt_path = BASE_CKPT_DIR / "last_resnet18_transformer_concat.pt"

    global_step = 0

    print("\nTraining details:")
    print("hardware:", HW_PROFILE)
    print("epochs:", cfg.epochs)
    print("steps_per_epoch:", len(train_loader))
    print("total_steps:", total_steps)
    print("train_batch_size:", cfg.train_batch_size)
    print("eval_batch_size:", cfg.eval_batch_size)
    print("num_workers:", cfg.num_workers)
    print("AMP fp16:", use_amp)
    print("monitor:", cfg.monitor_metric)
    print("params:", json.dumps(param_report, indent=2))

    for epoch in range(1, cfg.epochs + 1):
        model.train()

        epoch_loss = defaultdict(float)
        n_batches = 0

        optimizer.zero_grad(set_to_none=True)
        epoch_start = time.time()

        pbar = tqdm(train_loader, desc=f"epoch {epoch}/{cfg.epochs}", leave=True)

        for step, batch in enumerate(pbar):
            if cfg.max_train_batches is not None and step >= cfg.max_train_batches:
                break

            batch_start = time.time()

            batch = batch_to_device(batch, device)

            with torch.amp.autocast(device_type="cuda", enabled=use_amp):
                outputs = model(
                    images=batch["image"],
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                )

                loss_dict = compute_baseline_loss(outputs, batch, cfg)
                loss = loss_dict["loss"] / cfg.gradient_accumulation_steps

            scaler.scale(loss).backward()

            if (step + 1) % cfg.gradient_accumulation_steps == 0:
                scaler.unscale_(optimizer)
                grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)

                scaler.step(optimizer)
                scaler.update()

                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

                global_step += 1
            else:
                grad_norm = torch.tensor(0.0)

            for k, v in loss_dict.items():
                epoch_loss[k] += float(v.detach().cpu())

            n_batches += 1
            batch_time = time.time() - batch_start

            pbar.set_postfix({
                "loss": f"{epoch_loss['loss'] / max(1, n_batches):.4f}",
                "lr": f"{scheduler.get_last_lr()[0]:.2e}",
                "bt": f"{batch_time:.2f}s",
            })

        train_metrics = {f"train_{k}": v / max(1, n_batches) for k, v in epoch_loss.items()}

        val_metrics = evaluate_baseline_model(
            model,
            val_loader,
            device,
            cfg,
            max_batches=cfg.max_val_batches,
        )

        current_metric = val_metrics[cfg.monitor_metric]
        improved = current_metric > best_metric + cfg.min_delta

        epoch_time = time.time() - epoch_start

        row = {
            "epoch": epoch,
            "global_step": global_step,
            "epoch_time_sec": epoch_time,
            "lr_image": optimizer.param_groups[0]["lr"],
            "lr_other": optimizer.param_groups[1]["lr"],
            "best_metric_so_far": best_metric,
            **train_metrics,
            **{f"val_{k}": v for k, v in val_metrics.items()},
        }

        history.append(row)

        hist_df = pd.DataFrame(history)
        hist_df.to_csv(BASE_LOG_DIR / "resnet18_concat_history.csv", index=False)

        print("\n" + "-" * 90)
        print(f"Epoch {epoch}/{cfg.epochs} finished in {epoch_time/60:.2f} min")
        print(f"Train loss: {train_metrics.get('train_loss', float('nan')):.6f}")
        print(
            "Val:",
            f"acc={val_metrics['answer_accuracy']:.6f}",
            f"macro_f1={val_metrics['answer_macro_f1']:.6f}",
            f"weighted_f1={val_metrics['answer_weighted_f1']:.6f}",
            f"ece={val_metrics['answer_ece']:.6f}",
            f"loss={val_metrics['loss']:.6f}",
        )

        if improved:
            best_metric = current_metric
            best_epoch = epoch
            bad_epochs = 0

            save_baseline_checkpoint(
                best_ckpt_path,
                model,
                optimizer,
                scheduler,
                epoch,
                best_metric,
                history,
                cfg,
            )

            print(f"New best {cfg.monitor_metric}: {best_metric:.6f} at epoch {best_epoch}")
        else:
            bad_epochs += 1
            print(f"No improvement. bad_epochs={bad_epochs}/{cfg.early_stopping_patience}")

        save_baseline_checkpoint(
            last_ckpt_path,
            model,
            optimizer,
            scheduler,
            epoch,
            best_metric,
            history,
            cfg,
        )

        if cfg.save_every_epoch:
            save_baseline_checkpoint(
                BASE_CKPT_DIR / f"epoch_{epoch:03d}_resnet18_transformer_concat.pt",
                model,
                optimizer,
                scheduler,
                epoch,
                best_metric,
                history,
                cfg,
            )

        if cfg.early_stop and bad_epochs >= cfg.early_stopping_patience:
            print("Early stopping triggered.")
            break

    report = {
        "model_name": "ResNet18 + Bengali Transformer + Concat Fusion",
        "hardware_profile": HW_PROFILE,
        "epochs_completed": int(history[-1]["epoch"]) if history else 0,
        "best_epoch": int(best_epoch),
        "best_metric_name": cfg.monitor_metric,
        "best_metric": float(best_metric),
        "best_checkpoint": str(best_ckpt_path),
        "last_checkpoint": str(last_ckpt_path),
        "parameter_report": param_report,
        "config": asdict(cfg),
    }

    with open(BASE_REPORT_DIR / "resnet18_concat_training_report.json", "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)

    return report


print("Trainer ready.")

BASELINE TRAINER
Trainer ready.


In [8]:
# ============================================================
# FINAL FIXED Portion B7:
# Self-contained launch cell for ResNet18 + Transformer concat
# Fixes: NameError: build_dataloaders is not defined
# ============================================================

print("=" * 90)
print("LAUNCHING BASELINE TRAINING - FIXED SELF-CONTAINED CELL")
print("=" * 90)

import os
import gc
import json
import math
import time
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm

# ------------------------------------------------------------
# Required object checks
# ------------------------------------------------------------
required_objects = [
    "BASE_CFG",
    "train_ds",
    "val_ds",
    "test_ds",
    "build_baseline_model",
    "tokenizer",
    "label_mapping",
    "num_answers",
]

missing_objects = [name for name in required_objects if name not in globals()]
if missing_objects:
    raise NameError(
        "You must run B0, fixed B1, B2, and B3 before this cell. "
        f"Missing objects: {missing_objects}"
    )

# ------------------------------------------------------------
# Paths fallback
# ------------------------------------------------------------
PROJECT_DIR = Path("/kaggle/working/tinyagrivqa_bn") if Path("/kaggle/working").exists() else Path("./tinyagrivqa_bn")

BASE_DIR = globals().get("BASE_DIR", PROJECT_DIR / "portion_baseline_resnet18_concat")
BASE_CKPT_DIR = globals().get("BASE_CKPT_DIR", BASE_DIR / "checkpoints")
BASE_LOG_DIR = globals().get("BASE_LOG_DIR", BASE_DIR / "logs")
BASE_REPORT_DIR = globals().get("BASE_REPORT_DIR", BASE_DIR / "reports")
BASE_PRED_DIR = globals().get("BASE_PRED_DIR", BASE_DIR / "predictions")

for d in [BASE_DIR, BASE_CKPT_DIR, BASE_LOG_DIR, BASE_REPORT_DIR, BASE_PRED_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_GPUS = torch.cuda.device_count()

if "HW_PROFILE" not in globals():
    if torch.cuda.is_available():
        gpu_names = " ".join([torch.cuda.get_device_name(i).lower() for i in range(NUM_GPUS)])
        if NUM_GPUS >= 2 and "t4" in gpu_names:
            HW_PROFILE = "t4x2"
        elif "t4" in gpu_names:
            HW_PROFILE = "t4_single"
        else:
            HW_PROFILE = "cuda_generic"
    else:
        HW_PROFILE = "cpu"

if "BASE_SEED" not in globals():
    BASE_SEED = 42

def seed_everything(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(BASE_SEED)

# ------------------------------------------------------------
# Fix config fields if missing
# ------------------------------------------------------------
default_cfg_values = {
    "train_batch_size": 64,
    "eval_batch_size": 128,
    "gradient_accumulation_steps": 1,
    "num_workers": 2,
    "pin_memory": True,
    "epochs": 12,
    "lr": 2e-4,
    "image_encoder_lr": 5e-5,
    "weight_decay": 1e-4,
    "warmup_ratio": 0.06,
    "max_grad_norm": 1.0,
    "label_smoothing": 0.02,
    "mixed_precision": "fp16",
    "early_stop": True,
    "early_stopping_patience": 3,
    "min_delta": 0.001,
    "monitor_metric": "answer_macro_f1",
    "max_train_batches": None,
    "max_val_batches": None,
    "save_every_epoch": False,
}

for k, v in default_cfg_values.items():
    if not hasattr(BASE_CFG, k):
        setattr(BASE_CFG, k, v)

if HW_PROFILE == "cpu":
    BASE_CFG.train_batch_size = min(BASE_CFG.train_batch_size, 8)
    BASE_CFG.eval_batch_size = min(BASE_CFG.eval_batch_size, 16)
    BASE_CFG.num_workers = 0
    BASE_CFG.mixed_precision = "no"

# ------------------------------------------------------------
# Core utilities: this fixes build_dataloaders missing
# ------------------------------------------------------------
def build_dataloaders(cfg):
    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.train_batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=bool(cfg.pin_memory and torch.cuda.is_available()),
        drop_last=False,
        persistent_workers=bool(cfg.num_workers > 0),
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=bool(cfg.pin_memory and torch.cuda.is_available()),
        drop_last=False,
        persistent_workers=bool(cfg.num_workers > 0),
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=bool(cfg.pin_memory and torch.cuda.is_available()),
        drop_last=False,
        persistent_workers=bool(cfg.num_workers > 0),
    )

    return train_loader, val_loader, test_loader


def batch_to_device(batch, device):
    out = {}
    for k, v in batch.items():
        if torch.is_tensor(v):
            out[k] = v.to(device, non_blocking=True)
        else:
            out[k] = v
    return out


def compute_baseline_loss(outputs, batch, cfg):
    answer_loss = F.cross_entropy(
        outputs["answer_logits"],
        batch["answer_id"],
        label_smoothing=float(cfg.label_smoothing),
    )

    return {
        "loss": answer_loss,
        "answer_loss": answer_loss.detach(),
    }


def expected_calibration_error(probs, labels, n_bins=15):
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    correct = (predictions == labels).astype(np.float32)

    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0

    for i in range(n_bins):
        lo = bin_edges[i]
        hi = bin_edges[i + 1]

        if i == n_bins - 1:
            mask = (confidences >= lo) & (confidences <= hi)
        else:
            mask = (confidences >= lo) & (confidences < hi)

        if mask.sum() == 0:
            continue

        bin_acc = correct[mask].mean()
        bin_conf = confidences[mask].mean()
        bin_weight = mask.mean()

        ece += bin_weight * abs(bin_acc - bin_conf)

    return float(ece)


def compute_answer_metrics(logits_np, labels_np):
    logits_tensor = torch.tensor(logits_np, dtype=torch.float32)
    probs = torch.softmax(logits_tensor, dim=-1).numpy()
    preds = probs.argmax(axis=1)

    return {
        "answer_accuracy": float(accuracy_score(labels_np, preds)),
        "answer_macro_f1": float(f1_score(labels_np, preds, average="macro", zero_division=0)),
        "answer_weighted_f1": float(f1_score(labels_np, preds, average="weighted", zero_division=0)),
        "answer_ece": float(expected_calibration_error(probs, labels_np, n_bins=15)),
    }


def unwrap_model(model):
    return model.module if isinstance(model, torch.nn.DataParallel) else model


def count_parameters_safe(model):
    model = unwrap_model(model)
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {
        "total": int(total),
        "trainable": int(trainable),
        "total_m": float(total / 1e6),
        "trainable_m": float(trainable / 1e6),
    }


def build_optimizer(model, cfg):
    image_params = []
    other_params = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        clean_name = name[7:] if name.startswith("module.") else name

        if clean_name.startswith("image_encoder"):
            image_params.append(param)
        else:
            other_params.append(param)

    optimizer = torch.optim.AdamW(
        [
            {
                "params": image_params,
                "lr": float(cfg.image_encoder_lr),
                "weight_decay": float(cfg.weight_decay),
            },
            {
                "params": other_params,
                "lr": float(cfg.lr),
                "weight_decay": float(cfg.weight_decay),
            },
        ],
        betas=(0.9, 0.999),
        eps=1e-8,
    )

    print("Optimizer groups:")
    print(" image_encoder params:", sum(p.numel() for p in image_params))
    print(" other params:", sum(p.numel() for p in other_params))
    print(" image_encoder lr:", cfg.image_encoder_lr)
    print(" other lr:", cfg.lr)

    return optimizer


def build_cosine_scheduler(optimizer, num_training_steps, warmup_ratio=0.06):
    num_warmup_steps = int(num_training_steps * warmup_ratio)

    def lr_lambda(current_step):
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))

        progress = float(current_step - num_warmup_steps) / float(
            max(1, num_training_steps - num_warmup_steps)
        )
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def autocast_context(use_amp):
    if not use_amp:
        return torch.amp.autocast(device_type="cpu", enabled=False)

    try:
        return torch.amp.autocast(device_type="cuda", enabled=True)
    except Exception:
        return torch.cuda.amp.autocast(enabled=True)


def make_grad_scaler(use_amp):
    try:
        return torch.amp.GradScaler("cuda", enabled=use_amp)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=use_amp)


def evaluate_baseline_model(model, loader, device, cfg, max_batches=None):
    model.eval()

    all_logits = []
    all_labels = []
    loss_meter = defaultdict(float)
    n_batches = 0

    with torch.no_grad():
        pbar = tqdm(loader, desc="eval", leave=False)

        for step, batch in enumerate(pbar):
            if max_batches is not None and step >= max_batches:
                break

            batch = batch_to_device(batch, device)

            outputs = model(
                images=batch["image"],
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
            )

            loss_dict = compute_baseline_loss(outputs, batch, cfg)

            for k, v in loss_dict.items():
                loss_meter[k] += float(v.detach().cpu())

            all_logits.append(outputs["answer_logits"].detach().cpu())
            all_labels.append(batch["answer_id"].detach().cpu())

            n_batches += 1

    logits_np = torch.cat(all_logits, dim=0).numpy()
    labels_np = torch.cat(all_labels, dim=0).numpy()

    metrics = compute_answer_metrics(logits_np, labels_np)

    for k in list(loss_meter.keys()):
        metrics[k] = loss_meter[k] / max(1, n_batches)

    return metrics


def cfg_to_dict(cfg):
    try:
        from dataclasses import asdict, is_dataclass
        if is_dataclass(cfg):
            return asdict(cfg)
    except Exception:
        pass

    return {
        k: v for k, v in vars(cfg).items()
        if not k.startswith("_") and isinstance(v, (int, float, str, bool, type(None)))
    }


def save_baseline_checkpoint(path, model, optimizer, scheduler, epoch, best_metric, history, cfg):
    model_to_save = unwrap_model(model)

    torch.save(
        {
            "epoch": int(epoch),
            "model_state_dict": model_to_save.state_dict(),
            "optimizer_state_dict": optimizer.state_dict() if optimizer is not None else None,
            "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
            "best_metric": float(best_metric),
            "history": history,
            "model_config": {
                "model_name": "ResNet18TransformerConcatVQA",
                "image_backbone": "resnet18",
                "fusion": "concat",
                "vocab_size": len(tokenizer.token_to_id),
                "num_answers": int(num_answers),
                "max_len": int(getattr(cfg, "max_text_len", 48)),
                "image_pretrained": bool(getattr(cfg, "image_pretrained", False)),
                "text_dim": int(getattr(cfg, "text_dim", 256)),
                "text_layers": int(getattr(cfg, "text_layers", 3)),
                "text_heads": int(getattr(cfg, "text_heads", 4)),
                "text_ff_dim": int(getattr(cfg, "text_ff_dim", 768)),
                "fusion_dim": int(getattr(cfg, "fusion_dim", 512)),
                "dropout": float(getattr(cfg, "dropout", 0.10)),
                "pad_id": int(tokenizer.token_to_id[tokenizer.pad_token]),
            },
            "training_config": cfg_to_dict(cfg),
            "label_mapping": label_mapping,
        },
        path,
    )


# ------------------------------------------------------------
# Full trainer redefine: avoids dependency issues from old cells
# ------------------------------------------------------------
def train_resnet18_concat_baseline_fixed(cfg):
    seed_everything(BASE_SEED)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_loader, val_loader, _ = build_dataloaders(cfg)

    model = build_baseline_model(cfg)
    param_report = count_parameters_safe(model)

    model = model.to(device)

    if torch.cuda.device_count() > 1:
        print("Multiple GPUs detected. Using nn.DataParallel.")
        model = torch.nn.DataParallel(model)

    optimizer = build_optimizer(model, cfg)

    steps_per_epoch = math.ceil(len(train_loader) / max(1, int(cfg.gradient_accumulation_steps)))
    total_steps = steps_per_epoch * int(cfg.epochs)
    scheduler = build_cosine_scheduler(optimizer, total_steps, float(cfg.warmup_ratio))

    use_amp = bool(device.type == "cuda" and str(cfg.mixed_precision).lower() == "fp16")
    scaler = make_grad_scaler(use_amp)

    best_metric = -1.0
    best_epoch = -1
    bad_epochs = 0
    history = []

    best_ckpt_path = Path(BASE_CKPT_DIR) / "best_resnet18_transformer_concat.pt"
    last_ckpt_path = Path(BASE_CKPT_DIR) / "last_resnet18_transformer_concat.pt"

    global_step = 0

    print("\nTraining details:")
    print("hardware:", HW_PROFILE)
    print("device:", device)
    print("epochs:", cfg.epochs)
    print("train rows:", len(train_ds))
    print("val rows:", len(val_ds))
    print("steps_per_epoch:", len(train_loader))
    print("total_steps:", total_steps)
    print("train_batch_size:", cfg.train_batch_size)
    print("eval_batch_size:", cfg.eval_batch_size)
    print("num_workers:", cfg.num_workers)
    print("AMP fp16:", use_amp)
    print("monitor:", cfg.monitor_metric)
    print("params:", json.dumps(param_report, indent=2))

    for epoch in range(1, int(cfg.epochs) + 1):
        model.train()

        epoch_loss = defaultdict(float)
        n_batches = 0

        optimizer.zero_grad(set_to_none=True)
        epoch_start = time.time()

        pbar = tqdm(train_loader, desc=f"epoch {epoch}/{cfg.epochs}", leave=True)

        for step, batch in enumerate(pbar):
            if cfg.max_train_batches is not None and step >= cfg.max_train_batches:
                break

            batch = batch_to_device(batch, device)

            with autocast_context(use_amp):
                outputs = model(
                    images=batch["image"],
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                )

                loss_dict = compute_baseline_loss(outputs, batch, cfg)
                loss = loss_dict["loss"] / max(1, int(cfg.gradient_accumulation_steps))

            scaler.scale(loss).backward()

            if (step + 1) % max(1, int(cfg.gradient_accumulation_steps)) == 0:
                scaler.unscale_(optimizer)
                grad_norm = torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    float(cfg.max_grad_norm),
                )

                scaler.step(optimizer)
                scaler.update()

                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

                global_step += 1
            else:
                grad_norm = torch.tensor(0.0)

            for k, v in loss_dict.items():
                epoch_loss[k] += float(v.detach().cpu())

            n_batches += 1

            pbar.set_postfix({
                "loss": f"{epoch_loss['loss'] / max(1, n_batches):.4f}",
                "lr_img": f"{optimizer.param_groups[0]['lr']:.2e}",
                "lr_other": f"{optimizer.param_groups[1]['lr']:.2e}",
            })

        train_metrics = {
            f"train_{k}": v / max(1, n_batches)
            for k, v in epoch_loss.items()
        }

        val_metrics = evaluate_baseline_model(
            model,
            val_loader,
            device,
            cfg,
            max_batches=cfg.max_val_batches,
        )

        current_metric = val_metrics[str(cfg.monitor_metric)]
        improved = current_metric > best_metric + float(cfg.min_delta)

        epoch_time = time.time() - epoch_start

        row = {
            "epoch": int(epoch),
            "global_step": int(global_step),
            "epoch_time_sec": float(epoch_time),
            "lr_image": float(optimizer.param_groups[0]["lr"]),
            "lr_other": float(optimizer.param_groups[1]["lr"]),
            "best_metric_so_far": float(best_metric),
            **train_metrics,
            **{f"val_{k}": float(v) for k, v in val_metrics.items()},
        }

        history.append(row)

        hist_df = pd.DataFrame(history)
        hist_path = Path(BASE_LOG_DIR) / "resnet18_concat_history.csv"
        hist_df.to_csv(hist_path, index=False)

        print("\n" + "-" * 90)
        print(f"Epoch {epoch}/{cfg.epochs} finished in {epoch_time/60:.2f} min")
        print(f"Train loss: {train_metrics.get('train_loss', float('nan')):.6f}")
        print(
            "Val:",
            f"acc={val_metrics['answer_accuracy']:.6f}",
            f"macro_f1={val_metrics['answer_macro_f1']:.6f}",
            f"weighted_f1={val_metrics['answer_weighted_f1']:.6f}",
            f"ece={val_metrics['answer_ece']:.6f}",
            f"loss={val_metrics['loss']:.6f}",
        )

        if improved:
            best_metric = current_metric
            best_epoch = epoch
            bad_epochs = 0

            save_baseline_checkpoint(
                best_ckpt_path,
                model,
                optimizer,
                scheduler,
                epoch,
                best_metric,
                history,
                cfg,
            )

            print(f"New best {cfg.monitor_metric}: {best_metric:.6f} at epoch {best_epoch}")
        else:
            bad_epochs += 1
            print(f"No improvement. bad_epochs={bad_epochs}/{cfg.early_stopping_patience}")

        save_baseline_checkpoint(
            last_ckpt_path,
            model,
            optimizer,
            scheduler,
            epoch,
            best_metric,
            history,
            cfg,
        )

        if bool(cfg.save_every_epoch):
            save_baseline_checkpoint(
                Path(BASE_CKPT_DIR) / f"epoch_{epoch:03d}_resnet18_transformer_concat.pt",
                model,
                optimizer,
                scheduler,
                epoch,
                best_metric,
                history,
                cfg,
            )

        if bool(cfg.early_stop) and bad_epochs >= int(cfg.early_stopping_patience):
            print("Early stopping triggered.")
            break

    report = {
        "model_name": "ResNet18 + Bengali Transformer + Concat Fusion",
        "hardware_profile": HW_PROFILE,
        "epochs_completed": int(history[-1]["epoch"]) if history else 0,
        "best_epoch": int(best_epoch),
        "best_metric_name": str(cfg.monitor_metric),
        "best_metric": float(best_metric),
        "best_checkpoint": str(best_ckpt_path),
        "last_checkpoint": str(last_ckpt_path),
        "parameter_report": param_report,
        "config": cfg_to_dict(cfg),
    }

    report_path = Path(BASE_REPORT_DIR) / "resnet18_concat_training_report.json"
    with open(report_path, "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)

    print("\nSaved training report:", report_path)

    return report


# ------------------------------------------------------------
# Clear stale distributed state and CUDA cache
# ------------------------------------------------------------
try:
    import torch.distributed as dist
    if dist.is_available() and dist.is_initialized():
        dist.destroy_process_group()
        print("Destroyed stale distributed process group.")
except Exception as e:
    print("No active distributed process group:", repr(e))

for k in [
    "LOCAL_RANK",
    "RANK",
    "WORLD_SIZE",
    "MASTER_ADDR",
    "MASTER_PORT",
    "ACCELERATE_USE_CPU",
    "ACCELERATE_MIXED_PRECISION",
]:
    os.environ.pop(k, None)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("Available GPUs:", torch.cuda.device_count())
print("Hardware profile:", HW_PROFILE)
print("Epochs:", BASE_CFG.epochs)
print("Train batch size:", BASE_CFG.train_batch_size)
print("Eval batch size:", BASE_CFG.eval_batch_size)
print("Workers:", BASE_CFG.num_workers)
print("Early stopping:", BASE_CFG.early_stop)
print("Patience:", BASE_CFG.early_stopping_patience)

# Optional smoke/debug mode. Uncomment only for quick test.
# BASE_CFG.max_train_batches = 20
# BASE_CFG.max_val_batches = 10
# BASE_CFG.epochs = 1

# ------------------------------------------------------------
# Launch fixed training
# ------------------------------------------------------------
baseline_train_report = train_resnet18_concat_baseline_fixed(BASE_CFG)

print("=" * 90)
print("BASELINE TRAINING FINISHED")
print("=" * 90)
print(json.dumps(baseline_train_report, indent=2, ensure_ascii=False)[:4000])

LAUNCHING BASELINE TRAINING - FIXED SELF-CONTAINED CELL
Available GPUs: 2
Hardware profile: t4x2
Epochs: 12
Train batch size: 64
Eval batch size: 128
Workers: 2
Early stopping: True
Patience: 3
Multiple GPUs detected. Using nn.DataParallel.
Optimizer groups:
 image_encoder params: 11176512
 other params: 3256189
 image_encoder lr: 5e-05
 other lr: 0.0002

Training details:
hardware: t4x2
device: cuda
epochs: 12
train rows: 139455
val rows: 15522
steps_per_epoch: 2179
total_steps: 26148
train_batch_size: 64
eval_batch_size: 128
num_workers: 2
AMP fp16: True
monitor: answer_macro_f1
params: {
  "total": 14432701,
  "trainable": 14432701,
  "total_m": 14.432701,
  "trainable_m": 14.432701
}


epoch 1/12:   0%|          | 0/2179 [00:00<?, ?it/s]

eval:   0%|          | 0/122 [00:00<?, ?it/s]


------------------------------------------------------------------------------------------
Epoch 1/12 finished in 15.09 min
Train loss: 1.632426
Val: acc=0.683932 macro_f1=0.358394 weighted_f1=0.665608 ece=0.023431 loss=0.990698
New best answer_macro_f1: 0.358394 at epoch 1


epoch 2/12:   0%|          | 0/2179 [00:00<?, ?it/s]

...

eval:   0%|          | 0/122 [00:00<?, ?it/s]


------------------------------------------------------------------------------------------
Epoch 11/12 finished in 14.79 min
Train loss: 0.583274
Val: acc=0.767298 macro_f1=0.697611 weighted_f1=0.756704 ece=0.019553 loss=0.733056
New best answer_macro_f1: 0.697611 at epoch 11


epoch 12/12:   0%|          | 0/2179 [00:00<?, ?it/s]

eval:   0%|          | 0/122 [00:00<?, ?it/s]


------------------------------------------------------------------------------------------
Epoch 12/12 finished in 14.82 min
Train loss: 0.581303
Val: acc=0.768522 macro_f1=0.705762 weighted_f1=0.757732 ece=0.019345 loss=0.735796
New best answer_macro_f1: 0.705762 at epoch 12

Saved training report: /kaggle/working/tinyagrivqa_bn/portion_baseline_resnet18_concat/reports/resnet18_concat_training_report.json
BASELINE TRAINING FINISHED
{
  "model_name": "ResNet18 + Bengali Transformer + Concat Fusion",
  "hardware_profile": "t4x2",
  "epochs_completed": 12,
  "best_epoch": 12,
  "best_metric_name": "answer_macro_f1",
  "best_metric": 0.70576202889098,
  "best_checkpoint": "/kaggle/working/tinyagrivqa_bn/portion_baseline_resnet18_concat/checkpoints/best_resnet18_transformer_concat.pt",
  "last_checkpoint": "/kaggle/working/tinyagrivqa_bn/portion_baseline_resnet18_concat/checkpoints/last_resnet18_transformer_concat.pt",
  "parameter_report": {
    "total": 14432701,
    "trainable": 144327

In [9]:
# ============================================================
# Baseline Portion B8: Load best checkpoint and final evaluation
# ============================================================

print("=" * 90)
print("FINAL BASELINE EVALUATION")
print("=" * 90)

BEST_BASELINE_CKPT = BASE_CKPT_DIR / "best_resnet18_transformer_concat.pt"

if not BEST_BASELINE_CKPT.exists():
    raise FileNotFoundError(f"Best checkpoint not found: {BEST_BASELINE_CKPT}")


def safe_torch_load(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


ckpt = safe_torch_load(BEST_BASELINE_CKPT, map_location="cpu")

print("Best checkpoint:", BEST_BASELINE_CKPT)
print("Checkpoint epoch:", ckpt["epoch"])
print("Checkpoint best metric:", ckpt["best_metric"])

eval_model = build_baseline_model(BASE_CFG)
missing, unexpected = eval_model.load_state_dict(ckpt["model_state_dict"], strict=False)

print("Missing keys:", len(missing))
print("Unexpected keys:", len(unexpected))

eval_model = eval_model.to(DEVICE)

if torch.cuda.device_count() > 1:
    print("Using nn.DataParallel for evaluation.")
    eval_model = nn.DataParallel(eval_model)

_, val_loader_eval, test_loader_eval = build_dataloaders(BASE_CFG)

val_metrics = evaluate_baseline_model(
    eval_model,
    val_loader_eval,
    DEVICE,
    BASE_CFG,
    max_batches=None,
)

test_metrics = evaluate_baseline_model(
    eval_model,
    test_loader_eval,
    DEVICE,
    BASE_CFG,
    max_batches=None,
)

eval_report = {
    "model_name": "ResNet18 + Bengali Transformer + Concat Fusion",
    "best_checkpoint": str(BEST_BASELINE_CKPT),
    "best_epoch": int(ckpt["epoch"]),
    "best_val_metric": float(ckpt["best_metric"]),
    "val_metrics": val_metrics,
    "test_metrics": test_metrics,
    "parameter_report": count_parameters(unwrap_model(eval_model)),
    "config": asdict(BASE_CFG),
}

with open(BASE_REPORT_DIR / "resnet18_concat_eval_report.json", "w", encoding="utf-8") as f:
    json.dump(eval_report, f, ensure_ascii=False, indent=2)

print("\nValidation metrics:")
for k, v in val_metrics.items():
    print(f"{k}: {v:.6f}")

print("\nTest metrics:")
for k, v in test_metrics.items():
    print(f"{k}: {v:.6f}")

print("\nSaved eval report:")
print(BASE_REPORT_DIR / "resnet18_concat_eval_report.json")

FINAL BASELINE EVALUATION
Best checkpoint: /kaggle/working/tinyagrivqa_bn/portion_baseline_resnet18_concat/checkpoints/best_resnet18_transformer_concat.pt
Checkpoint epoch: 12
Checkpoint best metric: 0.70576202889098
Missing keys: 0
Unexpected keys: 0
Using nn.DataParallel for evaluation.


eval:   0%|          | 0/122 [00:00<?, ?it/s]

eval:   0%|          | 0/302 [00:00<?, ?it/s]


Validation metrics:
answer_accuracy: 0.768522
answer_macro_f1: 0.705762
answer_weighted_f1: 0.757732
answer_ece: 0.019345
loss: 0.735796
answer_loss: 0.735796

Test metrics:
answer_accuracy: 0.767680
answer_macro_f1: 0.659756
answer_weighted_f1: 0.756058
answer_ece: 0.022805
loss: 0.749801
answer_loss: 0.749801

Saved eval report:
/kaggle/working/tinyagrivqa_bn/portion_baseline_resnet18_concat/reports/resnet18_concat_eval_report.json


In [10]:
# ============================================================
# Baseline Portion B9: Export validation/test predictions
# ============================================================

print("=" * 90)
print("EXPORTING BASELINE PREDICTIONS")
print("=" * 90)

id_to_answer = {int(v): k for k, v in label_mapping["answer_to_id"].items()}


def export_baseline_predictions(model, loader, raw_df, split_name, device):
    model.eval()

    rows = []

    with torch.no_grad():
        for batch in tqdm(loader, desc=f"predict {split_name}"):
            batch_device = batch_to_device(batch, device)

            outputs = model(
                images=batch_device["image"],
                input_ids=batch_device["input_ids"],
                attention_mask=batch_device["attention_mask"],
            )

            probs = F.softmax(outputs["answer_logits"], dim=-1)
            conf, pred = probs.max(dim=-1)

            pred = pred.detach().cpu().numpy()
            conf = conf.detach().cpu().numpy()
            gold = batch["answer_id"].detach().cpu().numpy()

            bsz = len(gold)

            for i in range(bsz):
                rows.append({
                    "qa_id": batch["qa_id"][i],
                    "image_id": batch["image_id"][i],
                    "gold_answer_id": int(gold[i]),
                    "pred_answer_id": int(pred[i]),
                    "confidence": float(conf[i]),
                    "gold_answer_text": id_to_answer.get(int(gold[i]), "<missing>"),
                    "pred_answer_text": id_to_answer.get(int(pred[i]), "<missing>"),
                    "correct": int(int(gold[i]) == int(pred[i])),
                })

    pred_df = pd.DataFrame(rows)

    context_cols = [
        "qa_id",
        "resolved_image_path",
        "question_type",
        "bangla_question",
        "p2_answer_text",
        "p2_crop_label",
        "p2_disease_label",
        "p2_class_label",
    ]

    context_cols = [c for c in context_cols if c in raw_df.columns]
    context = raw_df[context_cols].drop_duplicates("qa_id")

    pred_df = pred_df.merge(context, on="qa_id", how="left")

    save_path = BASE_PRED_DIR / f"{split_name}_resnet18_concat_predictions.csv"
    pred_df.to_csv(save_path, index=False)

    print(split_name, "predictions:", pred_df.shape, "| saved:", save_path)

    return pred_df


val_pred_df = export_baseline_predictions(
    eval_model,
    val_loader_eval,
    val_df,
    "val",
    DEVICE,
)

test_pred_df = export_baseline_predictions(
    eval_model,
    test_loader_eval,
    test_df,
    "test",
    DEVICE,
)

print("\nValidation prediction accuracy from CSV:", val_pred_df["correct"].mean())
print("Test prediction accuracy from CSV:", test_pred_df["correct"].mean())

display(val_pred_df.head())
display(test_pred_df.head())

EXPORTING BASELINE PREDICTIONS


predict val:   0%|          | 0/122 [00:00<?, ?it/s]

val predictions: (15522, 15) | saved: /kaggle/working/tinyagrivqa_bn/portion_baseline_resnet18_concat/predictions/val_resnet18_concat_predictions.csv


predict test:   0%|          | 0/302 [00:00<?, ?it/s]

test predictions: (38632, 15) | saved: /kaggle/working/tinyagrivqa_bn/portion_baseline_resnet18_concat/predictions/test_resnet18_concat_predictions.csv

Validation prediction accuracy from CSV: 0.7685220976678263
Test prediction accuracy from CSV: 0.767679643818596


,qa_id,image_id,gold_answer_id,pred_answer_id,confidence,gold_answer_text,pred_answer_text,correct,resolved_image_path,question_type,bangla_question,p2_answer_text,p2_crop_label,p2_disease_label,p2_class_label
0,val_000003_f629cf9b,image_000002.JPG,12,12,0.730487,পাতাটি সুস্থ; নির্দিষ্ট রোগের লক্ষণ দেখা যাচ্ছ...,পাতাটি সুস্থ; নির্দিষ্ট রোগের লক্ষণ দেখা যাচ্ছ...,1,/kaggle/working/tinyagrivqa_bn/PlantVillageVQA...,Specific Disease Identification,এই পাতায় কোন রোগ দেখা যাচ্ছে?,পাতাটি সুস্থ; নির্দিষ্ট রোগের লক্ষণ দেখা যাচ্ছ...,unknown,healthy,unknown_healthy
1,val_000007_fd844bf2,image_000005.JPG,8,8,0.982620,দৃশ্যমান লক্ষণগুলো সম্ভবত টমেটোর ইয়েলো লিফ কার...,দৃশ্যমান লক্ষণগুলো সম্ভবত টমেটোর ইয়েলো লিফ কার...,1,/kaggle/working/tinyagrivqa_bn/PlantVillageVQA...,Causal Reasoning,এই রোগের সম্ভাব্য কারণ কী?,দৃশ্যমান লক্ষণগুলো সম্ভবত টমেটোর ইয়েলো লিফ কার...,tomato,tomato_yellow_leaf_curl_virus,tomato_yellow_leaf_curl_virus
2,val_000008_53c1b0e9,image_000005.JPG,7,7,0.983266,ছবিতে পাতায় টমেটোর ইয়েলো লিফ কার্ল ভাইরাস-এর ...,ছবিতে পাতায় টমেটোর ইয়েলো লিফ কার্ল ভাইরাস-এর ...,1,/kaggle/working/tinyagrivqa_bn/PlantVillageVQA...,Comprehensive Description,এই পাতায় কী ধরনের দৃশ্যমান লক্ষণ দেখা যাচ্ছে?,ছবিতে পাতায় টমেটোর ইয়েলো লিফ কার্ল ভাইরাস-এর ...,tomato,tomato_yellow_leaf_curl_virus,tomato_yellow_leaf_curl_virus
3,val_000009_ebbd419c,image_000005.JPG,15,15,0.982192,রোগ না থাকলে টমেটোর ইয়েলো লিফ কার্ল ভাইরাস-এর ...,রোগ না থাকলে টমেটোর ইয়েলো লিফ কার্ল ভাইরাস-এর ...,1,/kaggle/working/tinyagrivqa_bn/PlantVillageVQA...,Counterfactual Reasoning,রোগ না থাকলে পাতার চেহারা কেমন হতো?,রোগ না থাকলে টমেটোর ইয়েলো লিফ কার্ল ভাইরাস-এর ...,tomato,tomato_yellow_leaf_curl_virus,tomato_yellow_leaf_curl_virus
4,val_000010_4aea63dd,image_000005.JPG,0,0,0.656924,না।,না।,1,/kaggle/working/tinyagrivqa_bn/PlantVillageVQA...,General Health Assessment,পাতাটি কি সুস্থ?,না।,tomato,tomato_yellow_leaf_curl_virus,tomato_yellow_leaf_curl_virus


,qa_id,image_id,gold_answer_id,pred_answer_id,confidence,gold_answer_text,pred_answer_text,correct,resolved_image_path,question_type,bangla_question,p2_answer_text,p2_crop_label,p2_disease_label,p2_class_label
0,test_130455_8b6efc14,image_044358.JPG,8,8,0.986893,দৃশ্যমান লক্ষণগুলো সম্ভবত টমেটোর ইয়েলো লিফ কার...,দৃশ্যমান লক্ষণগুলো সম্ভবত টমেটোর ইয়েলো লিফ কার...,1,/kaggle/working/tinyagrivqa_bn/PlantVillageVQA...,Causal Reasoning,পাতার এই সমস্যার কারণ কী হতে পারে?,দৃশ্যমান লক্ষণগুলো সম্ভবত টমেটোর ইয়েলো লিফ কার...,tomato,tomato_yellow_leaf_curl_virus,tomato_yellow_leaf_curl_virus
1,test_130456_43e23515,image_044358.JPG,7,7,0.987230,ছবিতে পাতায় টমেটোর ইয়েলো লিফ কার্ল ভাইরাস-এর ...,ছবিতে পাতায় টমেটোর ইয়েলো লিফ কার্ল ভাইরাস-এর ...,1,/kaggle/working/tinyagrivqa_bn/PlantVillageVQA...,Comprehensive Description,এই পাতায় কী ধরনের দৃশ্যমান লক্ষণ দেখা যাচ্ছে?,ছবিতে পাতায় টমেটোর ইয়েলো লিফ কার্ল ভাইরাস-এর ...,tomato,tomato_yellow_leaf_curl_virus,tomato_yellow_leaf_curl_virus
2,test_130457_76cd2c91,image_044358.JPG,0,0,0.651671,না।,না।,1,/kaggle/working/tinyagrivqa_bn/PlantVillageVQA...,General Health Assessment,এই পাতার স্বাস্থ্য অবস্থা কী?,না।,tomato,tomato_yellow_leaf_curl_virus,tomato_yellow_leaf_curl_virus
3,test_130458_4a03564f,image_044358.JPG,2,2,0.570938,এটি টমেটো গাছের পাতা।,এটি টমেটো গাছের পাতা।,1,/kaggle/working/tinyagrivqa_bn/PlantVillageVQA...,Plant Species Identification,এই পাতাটি কোন উদ্ভিদ প্রজাতির?,এটি টমেটো গাছের পাতা।,tomato,tomato_yellow_leaf_curl_virus,tomato_yellow_leaf_curl_virus
4,test_130459_f4e6b4ed,image_044358.JPG,6,6,0.986717,সম্ভাব্য নির্দিষ্ট রোগ হলো টমেটোর ইয়েলো লিফ কা...,সম্ভাব্য নির্দিষ্ট রোগ হলো টমেটোর ইয়েলো লিফ কা...,1,/kaggle/working/tinyagrivqa_bn/PlantVillageVQA...,Specific Disease Identification,এই উদ্ভিদের পাতায় কোন রোগ শনাক্ত করা যায়?,সম্ভাব্য নির্দিষ্ট রোগ হলো টমেটোর ইয়েলো লিফ কা...,tomato,tomato_yellow_leaf_curl_virus,tomato_yellow_leaf_curl_virus


In [11]:
# ============================================================
# Baseline Portion B10: Paper-ready result row
# ============================================================

print("=" * 90)
print("PAPER-READY BASELINE RESULT")
print("=" * 90)

with open(BASE_REPORT_DIR / "resnet18_concat_eval_report.json", "r", encoding="utf-8") as f:
    baseline_eval = json.load(f)

test_m = baseline_eval["test_metrics"]
val_m = baseline_eval["val_metrics"]
param_m = baseline_eval["parameter_report"]

baseline_row = {
    "model": "ResNet18 + Transformer concat",
    "input": "Image + Bengali question",
    "image_encoder": "ResNet18",
    "text_encoder": "3-layer Bengali Transformer",
    "fusion": "Concatenation",
    "auxiliary_heads": "No",
    "val_accuracy": val_m["answer_accuracy"],
    "val_macro_f1": val_m["answer_macro_f1"],
    "val_weighted_f1": val_m["answer_weighted_f1"],
    "val_ece": val_m["answer_ece"],
    "test_accuracy": test_m["answer_accuracy"],
    "test_macro_f1": test_m["answer_macro_f1"],
    "test_weighted_f1": test_m["answer_weighted_f1"],
    "test_ece": test_m["answer_ece"],
    "params_m": param_m["total_m"],
    "best_epoch": baseline_eval["best_epoch"],
}

baseline_summary_df = pd.DataFrame([baseline_row])

summary_path = BASE_REPORT_DIR / "baseline_summary_for_paper.csv"
baseline_summary_df.to_csv(summary_path, index=False)

print("Saved:", summary_path)
display(baseline_summary_df)

print("\nLaTeX row:")
print(
    f"ResNet18 + Transformer concat & Image+Text & "
    f"{100*baseline_row['test_accuracy']:.2f} & "
    f"{100*baseline_row['test_macro_f1']:.2f} & "
    f"{100*baseline_row['test_weighted_f1']:.2f} & "
    f"{100*baseline_row['test_ece']:.2f} \\\\"
)

PAPER-READY BASELINE RESULT
Saved: /kaggle/working/tinyagrivqa_bn/portion_baseline_resnet18_concat/reports/baseline_summary_for_paper.csv


,model,input,image_encoder,text_encoder,fusion,auxiliary_heads,val_accuracy,val_macro_f1,val_weighted_f1,val_ece,test_accuracy,test_macro_f1,test_weighted_f1,test_ece,params_m,best_epoch
0,ResNet18 + Transformer concat,Image + Bengali question,ResNet18,3-layer Bengali Transformer,Concatenation,No,0.768522,0.705762,0.757732,0.019345,0.76768,0.659756,0.756058,0.022805,14.432701,12



LaTeX row:
ResNet18 + Transformer concat & Image+Text & 76.77 & 65.98 & 75.61 & 2.28 \\
